# Kia Sportage: reserved tests and paired videos
Reproduces frozen local evaluation with saved speed-blind measurements and registration caches. Eight development references; 25 reserved tests. No test reference is used by calibration. Renders 33 calibration and 33 speed videos. Vehicle-template scale is reference-assisted.

In [ ]:
from pathlib import Path
import json,subprocess,sys,shutil,hashlib
sources={'calibrate_speed.py': '"""Experimental local road rectification from learned cuboids and explicit car-size priors.\n\nIndependent implementation of rectangle/aspect/rigidity fitting, inspired by\nSochor et al. (2017) and the UrbanOmniDetect (2026) ground-footprint approach.\nRanges are sensitivity envelopes, not confidence intervals or accuracy claims.\n"""\nfrom pathlib import Path\nfrom collections import defaultdict\nimport csv\nimport json\nimport math\nimport os\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\n\nROOT = Path(os.environ.get(\'FYP_SPEED_ROOT\',str(Path(__file__).resolve().parent)))\ndownloaded_geometry = ROOT / \'outputs\' / \'geometry\'\ndefault_geometry = downloaded_geometry if downloaded_geometry.exists() else ROOT / \'output\' / \'inputs\' / \'geometry\'\nGEOMETRY = Path(os.environ.get(\'FYP_GEOMETRY_DIR\', str(default_geometry)))\nOUT = Path(os.environ.get(\'FYP_SPEED_OUTPUT\',str(ROOT / \'outputs\' / \'speed_experiment\')))\nLENGTH_M, WIDTH_M = 4.5, 1.8\nDIMENSION_SCENARIOS = [(4.5,1.8),(3.2,1.45),(3.2,2.0),(5.2,1.45),(5.2,2.0)]\n\n\ndef project(points, H):\n    p = np.asarray(points, dtype=float)\n    flat = p.reshape(-1,2)\n    result = np.c_[flat,np.ones(len(flat))] @ H.T\n    denominator = result[:,2]\n    safe = np.where(np.abs(denominator)<1e-8, np.where(denominator>=0,1e-8,-1e-8),denominator)\n    return (result[:,:2]/safe[:,None]).reshape(p.shape)\n\n\ndef compose(theta, normalizer):\n    p,q,a,b = theta\n    return np.array([[np.exp(a),b,0],[0,1,0],[p,q,1]]) @ normalizer\n\n\ndef measures(quads):\n    edges = np.roll(quads,-1,axis=1)-quads\n    sizes = np.linalg.norm(edges,axis=2)+1e-9\n    unit = edges/sizes[:,:,None]\n    cosines = np.sum(unit*np.roll(unit,-1,axis=1),axis=2)\n    opposite = unit[:,:2,0]*unit[:,2:,1]-unit[:,:2,1]*unit[:,2:,0]\n    lengths=(sizes[:,0]+sizes[:,2])/2\n    widths=(sizes[:,1]+sizes[:,3])/2\n    return cosines,opposite,lengths,widths\n\n\ndef fit_mapping(records, width, height, length_m=LENGTH_M, width_m=WIDTH_M, fast=False, legacy_normalization=False):\n    quads=np.array([r[\'quad\'] for r in records],dtype=float)\n    ids=np.array([r[\'track_id\'] for r in records])\n    unique=np.unique(ids)\n    center=np.median(quads.reshape(-1,2),axis=0)\n    normalizer=np.array([[2/width,0,-2*center[0]/width],[0,2/width,-2*center[1]/width],[0,0,1]])\n    if legacy_normalization:\n        normalizer=np.array([[2/width,0,-1],[0,2/width,-height/width],[0,0,1]])\n    weights=np.array([1/math.sqrt(np.sum(ids==i)) for i in ids])\n    weights/=np.mean(weights)\n    groups=[np.flatnonzero(ids==i) for i in unique]\n\n    def residual(theta):\n        H=compose(theta,normalizer)\n        world=project(quads,H)\n        angles,parallel,lengths,widths=measures(world)\n        rigidity=np.zeros(len(records))\n        for group in groups:\n            rigidity[group]=np.log(lengths[group]/np.median(lengths[group]))\n        shape=np.log((lengths/widths)/(length_m/width_m))\n        geometric=np.c_[angles,parallel*0.5,shape*0.65,rigidity*0.5]*weights[:,None]\n        den=(np.c_[quads.reshape(-1,2),np.ones(len(quads)*4)]@H.T)[:,2]\n        pole=np.maximum(0,0.08-den)*15\n        return np.r_[geometric.ravel(),pole]\n\n    q_starts=([0,1,2] if not fast else [1,2]) if legacy_normalization else ([0,2,5,10,20] if not fast else [2,10])\n    aspect_starts=([0,0.7] if not fast else [0]) if legacy_normalization else ([-3,0] if not fast else [-2])\n    starts=[(0,q,a,0) for q in q_starts for a in aspect_starts]\n    fits=[]\n    for theta in starts:\n        bound=4 if legacy_normalization else 100\n        shape_bound=2.5 if legacy_normalization else 7\n        shear_bound=4 if legacy_normalization else 30\n        result=least_squares(residual,theta,bounds=([-bound,-bound,-shape_bound,-shear_bound],[bound,bound,shape_bound,shear_bound]),\n                             loss=\'soft_l1\',f_scale=0.12,max_nfev=500)\n        fits.append((float(np.mean(residual(result.x)**2)),result))\n    _,result=min(fits,key=lambda item:item[0])\n    H=compose(result.x,normalizer)\n    world=project(quads,H)\n    _,_,lengths,_=measures(world)\n    # Give each track one vote; correlated video frames do not count as independent cars.\n    scale=length_m/np.median([np.median(lengths[g]) for g in groups])\n    H=np.diag([scale,scale,1])@H\n    world=project(quads,H)\n    directions=world[:,1]-world[:,0]\n    directions/=np.linalg.norm(directions,axis=1)[:,None]\n    directions[directions[:,1]<0]*=-1\n    direction=np.median(directions,axis=0)\n    angle=math.pi/2-math.atan2(direction[1],direction[0])\n    rotation=np.array([[math.cos(angle),-math.sin(angle),0],[math.sin(angle),math.cos(angle),0],[0,0,1]])\n    H=rotation@H\n    center=np.median(project(quads,H).mean(axis=1),axis=0)\n    H=np.array([[1,0,-center[0]],[0,1,-center[1]],[0,0,1]])@H\n    H/=np.linalg.norm(H[2,:2]) if np.linalg.norm(H[2,:2])>0 else H[2,2]\n    angle_error,_,lengths,widths=measures(project(quads,H))\n    singular=np.linalg.svd(result.jac,compute_uv=False)\n    diagnostics={\'fit_track_ids\':[int(i) for i in unique],\'fit_observations\':len(records),\n       \'median_angle_error_deg\':float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angle_error),0,1))))),\n       \'median_length_m\':float(np.median(lengths)),\'median_width_m\':float(np.median(widths)),\n       \'parameter_condition_number\':float(singular[0]/max(singular[-1],1e-12)),\n       \'optimizer_success\':bool(result.success),\'theta\':result.x.tolist(),\n       \'dimension_prior_length_m\':length_m,\'dimension_prior_width_m\':width_m}\n    return H,diagnostics\n\n\ndef prepare(observations, indices, width, height, longitudinal_order=\'checkpoint\'):\n    by_id=defaultdict(list)\n    for r in observations:by_id[r[\'track_id\']].append(r)\n    records=[]\n    for tid,group in by_id.items():\n        group.sort(key=lambda r:r[\'time_s\'])\n        centers=np.array([[(r[\'baseline_box\'][0]+r[\'baseline_box\'][2])/2,\n                           (r[\'baseline_box\'][1]+r[\'baseline_box\'][3])/2] for r in group])\n        if len(group)<8 or np.linalg.norm(centers[-1]-centers[0])<80:continue\n        side=\'left\' if np.median(centers[:,0])<width/2 else \'right\'\n        for j,r in enumerate(group):\n            p=np.array(r[\'keypoints\'])[indices]\n            if p.shape[1]>2 and np.min(p[:,2])<0.30:continue\n            if r[\'geometry_confidence\']<0.25 or r[\'association_iou\']<0.40:continue\n            quad=p[:,:2]\n            if not np.isfinite(quad).all():continue\n            if longitudinal_order==\'motion\':\n                quad=quad[np.argsort(np.arctan2(*(quad-quad.mean(axis=0))[:,::-1].T))]\n            if not cv2.isContourConvex(quad.astype(np.float32)):continue\n            if abs(cv2.contourArea(quad.astype(np.float32)))<300:continue\n            if np.min(np.linalg.norm(np.roll(quad,-1,axis=0)-quad,axis=1))<8:continue\n            cx,cy=quad.mean(axis=0)\n            if not (0.20*height<cy<0.82*height and 0.02*width<cx<0.98*width):continue\n            if np.min(quad[:,0])<4 or np.max(quad[:,0])>width-4 or np.max(quad[:,1])>height-4:continue\n            if longitudinal_order==\'motion\':\n                lo=max(0,j-3);hi=min(len(group)-1,j+3)\n                motion=centers[hi]-centers[lo]\n                if np.linalg.norm(motion)<5:continue\n                edge=np.roll(quad,-1,axis=0)-quad\n                alignment=np.abs(edge@motion)/(np.linalg.norm(edge,axis=1)*np.linalg.norm(motion)+1e-9)\n                scores=[np.mean(alignment[[0,2]]),np.mean(alignment[[1,3]])]\n                if abs(scores[0]-scores[1])<0.08:continue\n                if scores[1]>scores[0]:quad=np.roll(quad,-1,axis=0)\n            elif longitudinal_order!=\'checkpoint\':\n                raise ValueError(\'Unknown longitudinal corner-order method.\')\n            records.append({\'track_id\':int(tid),\'time_s\':r[\'time_s\'],\'frame_index\':r[\'frame_index\'],\n                            \'side\':side,\'quad\':quad.tolist()})\n    return records\n\n\ndef estimate_track(records,H):\n    times=np.array([r[\'time_s\'] for r in records],dtype=float)\n    quads=np.array([r[\'quad\'] for r in records],dtype=float)\n    world=project(quads,H)\n    centers=world.mean(axis=1)\n    t=times-times.mean()\n    initial=np.r_[np.polyfit(t,centers[:,0],1),np.polyfit(t,centers[:,1],1)]\n    def residual(p):return np.r_[(p[0]*t+p[1]-centers[:,0]),(p[2]*t+p[3]-centers[:,1])]\n    fitted=least_squares(residual,initial,loss=\'soft_l1\',f_scale=0.4)\n    speed=float(np.hypot(fitted.x[0],fitted.x[2])*3.6)\n    rms=float(np.sqrt(np.mean(residual(fitted.x)**2)))\n    angles,_,lengths,widths=measures(world)\n    length_cv=float(1.4826*np.median(np.abs(lengths-np.median(lengths)))/np.median(lengths))\n    angle_error=float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angles),0,1)))))\n    midpoint=(times[0]+times[-1])/2\n    half_speeds=[]\n    for mask in [times<=midpoint,times>midpoint]:\n        if mask.sum()>=4:\n            velocity=np.array([np.polyfit(times[mask]-times[mask].mean(),centers[mask,k],1)[0] for k in [0,1]])\n            half_speeds.append(float(np.linalg.norm(velocity)*3.6))\n    drift=abs(half_speeds[0]-half_speeds[1])/max(speed,1) if len(half_speeds)==2 else None\n    return {\'experimental_kmh\':speed,\'trajectory_fit_rms_m\':rms,\'length_relative_mad\':length_cv,\n            \'angle_error_deg\':angle_error,\'half_span_speed_relative_difference\':drift,\n            \'median_inferred_length_m\':float(np.median(lengths)),\'median_inferred_width_m\':float(np.median(widths))}\n\n\ndef sample_balanced(records, maximum=10):\n    groups=defaultdict(list)\n    for r in records:groups[r[\'track_id\']].append(r)\n    return [group[i] for group in groups.values() for i in np.unique(np.linspace(0,len(group)-1,min(len(group),maximum)).astype(int))]\n\n\ndef fit_road_mappings(records, report):\n    width,height=report[\'width\'],report[\'height\']\n    calibration={}\n    for side in [\'left\',\'right\']:\n        local=[r for r in records if r[\'side\']==side]\n        ids=sorted(set(r[\'track_id\'] for r in local))\n        if len(ids)<3:\n            calibration[side]={\'accepted\':False,\'reason\':\'Fewer than three qualifying car tracks\',\'track_ids\':ids}\n            continue\n        # Deterministic holdout by track identity; held-out predicted boxes are consistency evidence only.\n        held_ids=ids[::3]\n        fit_ids=[i for i in ids if i not in held_ids]\n        if len(fit_ids)<3:fit_ids=ids;held_ids=[]\n        fitting=sample_balanced([r for r in local if r[\'track_id\'] in fit_ids])\n        H,diagnostics=fit_mapping(fitting,width,height)\n        alternatives=[]\n        for length_m,width_m in DIMENSION_SCENARIOS[1:]:\n            alternative,_=fit_mapping(fitting,width,height,length_m,width_m,fast=True)\n            alternatives.append(alternative)\n        rng=np.random.default_rng(42)\n        for _ in range(10):\n            selected=set(rng.choice(fit_ids,size=max(3,round(len(fit_ids)*0.75)),replace=False).tolist())\n            subset=[r for r in fitting if r[\'track_id\'] in selected]\n            alternative,_=fit_mapping(subset,width,height,fast=True)\n            alternatives.append(alternative)\n        held=[r for r in local if r[\'track_id\'] in held_ids]\n        held_angle=None\n        if held:\n            angles,_,_,_=measures(project(np.array([r[\'quad\'] for r in held]),H))\n            held_angle=float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angles),0,1)))))\n        accepted=diagnostics[\'median_angle_error_deg\']<20 and (held_angle is None or held_angle<25)\n        accepted=accepted and report[\'ground_order_vote_agreement\']>=0.8 and diagnostics[\'parameter_condition_number\']<1e5\n        diagnostics.update({\'accepted\':bool(accepted),\'held_out_track_ids\':held_ids,\n          \'held_out_median_angle_error_deg\':held_angle,\'H_image_to_experimental_meters\':H.tolist(),\n          \'alternative_mappings\':[a.tolist() for a in alternatives],\n          \'interpretation\':\'Approximate local plane and dimension-prior mapping; no measured accuracy.\'})\n        calibration[side]=diagnostics\n        diagnostics[\'support_hull_image\'] = cv2.convexHull(\n            np.array([r[\'quad\'] for r in local], dtype=np.float32).reshape(-1, 2)).reshape(-1, 2).tolist()\n    return calibration\n\n\ndef calculate_speed_rows(records, calibration):\n    speed_rows=[]\n    for side in [\'left\',\'right\']:\n        diagnostics=calibration.get(side,{})\n        if not diagnostics.get(\'accepted\'):\n            continue\n        H=np.array(diagnostics[\'H_image_to_experimental_meters\'])\n        alternatives=[np.array(a) for a in diagnostics[\'alternative_mappings\']]\n        hull=np.array(diagnostics[\'support_hull_image\'], dtype=np.float32)\n        # Restrict reuse to the road region supported by the calibration footprints.\n        local=[r for r in records if r[\'side\']==side and all(\n            cv2.pointPolygonTest(hull, (float(p[0]),float(p[1])), True)>=-0.01 for p in r[\'quad\'])]\n        ids=sorted(set(r[\'track_id\'] for r in local))\n        for tid in ids:\n            group=sorted([r for r in local if r[\'track_id\']==tid],key=lambda r:r[\'time_s\'])\n            if len(group)<8 or group[-1][\'time_s\']-group[0][\'time_s\']<1.2:continue\n            nominal=estimate_track(group,H)\n            values=[nominal[\'experimental_kmh\']]+[estimate_track(group,a)[\'experimental_kmh\'] for a in alternatives]\n            low,high=float(min(values)),float(max(values))\n            sensitivity=(high-low)/max(nominal[\'experimental_kmh\'],1)\n            reasons=[]\n            if nominal[\'length_relative_mad\']>0.25:reasons.append(\'predicted_size_inconsistent\')\n            if nominal[\'trajectory_fit_rms_m\']>0.8:reasons.append(\'trajectory_not_consistent_with_local_plane\')\n            if nominal[\'angle_error_deg\']>25:reasons.append(\'footprint_not_rectangular_after_mapping\')\n            if nominal[\'half_span_speed_relative_difference\'] is not None and nominal[\'half_span_speed_relative_difference\']>0.35:reasons.append(\'speed_changes_or_geometry_drift_across_span\')\n            if sensitivity>0.8:reasons.append(\'high_calibration_sensitivity\')\n            speed_rows.append({\'track_id\':tid,\'side\':side,\'time_start_s\':group[0][\'time_s\'],\n              \'time_end_s\':group[-1][\'time_s\'],\'geometry_observations\':len(group),\n              \'quality_passed\':not bool(reasons),\'quality_reasons\':\';\'.join(reasons),\n              \'dimension_and_fit_sensitivity_low_kmh\':low,\'dimension_and_fit_sensitivity_high_kmh\':high,\n              **nominal})\n    return speed_rows\n\n\ndef run(calibration=None, geometry_dir=None, output_dir=None, calibration_reused=None):\n    geometry_dir=Path(geometry_dir) if geometry_dir is not None else GEOMETRY\n    output_dir=Path(output_dir) if output_dir is not None else OUT\n    output_dir.mkdir(parents=True,exist_ok=True)\n    report=json.loads((geometry_dir/\'geometry_run_report.json\').read_text())\n    observations=json.loads((geometry_dir/\'observations.json\').read_text())\n    records=prepare(observations,report[\'ground_indices\'],report[\'width\'],report[\'height\'])\n    reused=(calibration is not None) if calibration_reused is None else calibration_reused\n    if calibration is None:\n        calibration=fit_road_mappings(records, report)\n    before_support_filter=len(records)\n    records=[r for r in records if calibration.get(r[\'side\'],{}).get(\'accepted\') and all(\n        cv2.pointPolygonTest(np.array(calibration[r[\'side\']][\'support_hull_image\'],dtype=np.float32),\n                             (float(p[0]),float(p[1])),True)>=-0.01 for p in r[\'quad\'])]\n    speed_rows=calculate_speed_rows(records, calibration)\n    (output_dir/\'filtered_ground_footprints.json\').write_text(json.dumps(records),encoding=\'utf-8\')\n    (output_dir/\'calibration.json\').write_text(json.dumps(calibration,indent=2),encoding=\'utf-8\')\n    fields=[\'track_id\',\'side\',\'time_start_s\',\'time_end_s\',\'geometry_observations\',\n            \'quality_passed\',\'quality_reasons\',\'dimension_and_fit_sensitivity_low_kmh\',\n            \'dimension_and_fit_sensitivity_high_kmh\',\'experimental_kmh\',\'trajectory_fit_rms_m\',\n            \'length_relative_mad\',\'angle_error_deg\',\'half_span_speed_relative_difference\',\n            \'median_inferred_length_m\',\'median_inferred_width_m\']\n    for name, rows in [(\'experimental_speeds.csv\',speed_rows),\n                       (\'qualified_experimental_speeds.csv\',[r for r in speed_rows if r[\'quality_passed\']])]:\n        with (output_dir/name).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n    api_results=[{\'track_id\':r[\'track_id\'],\'carriageway\':r[\'side\'],\n      \'time_interval_s\':[r[\'time_start_s\'],r[\'time_end_s\']],\n      \'experimental_kmh\':r[\'experimental_kmh\'],\n      \'estimated_kmh\':r[\'experimental_kmh\'],\n      \'sensitivity_range_kmh\':[r[\'dimension_and_fit_sensitivity_low_kmh\'],r[\'dimension_and_fit_sensitivity_high_kmh\']],\n      \'range_type\':\'dimension_and_calibration_sensitivity\',\'accuracy_validated\':False,\n      \'method\':\'learned_3d_ground_footprints_with_assumed_vehicle_dimensions\'}\n      for r in speed_rows if r[\'quality_passed\']]\n    (output_dir/\'vehicle_speed_estimates.json\').write_text(json.dumps(api_results,indent=2),encoding=\'utf-8\')\n    result={\'calibration_reused\':reused,\'source_geometry_observations\':len(observations),\'filtered_ground_footprints\':len(records),\n      \'footprints_outside_usable_calibration\':before_support_filter-len(records),\n      \'geometry_model_filename\':report[\'model_filename\'],\'geometry_model_sha256\':report[\'model_sha256\'],\n      \'ground_corner_indices\':report[\'ground_indices\'],\n      \'experimental_track_estimates\':len(speed_rows),\'quality_passed_track_estimates\':sum(r[\'quality_passed\'] for r in speed_rows),\n      \'nominal_car_length_m\':LENGTH_M,\'nominal_car_width_m\':WIDTH_M,\'dimension_scenarios_m\':DIMENSION_SCENARIOS,\n      \'range_type\':\'Dimension-prior and track-subset sensitivity envelope. Not a statistical confidence interval.\',\n      \'scope\':\'Local visible road regions on each carriageway; passenger-car hypotheses only.\',\n      \'accuracy_validated\':False,\'speed_units\':\'km/h under assumed vehicle dimensions\',\n      \'limitations\':[\'No independent speed or road-distance ground truth.\',\n        \'Ground corners are model predictions; hidden corners can be wrong.\',\n        \'A local planar road and approximately constant speed over each fitted span are assumed.\',\n        \'Dimension priors are explicit analyst assumptions, not recognized make/model dimensions.\',\n        \'Sensitivity range does not bound unmodelled errors such as lens distortion or road grade.\']}\n    (output_dir/\'speed_run_report.json\').write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n    print(json.dumps(result,indent=2))\n    for r in speed_rows:print(r[\'track_id\'],r[\'side\'],round(r[\'experimental_kmh\'],1),r[\'quality_passed\'],r[\'quality_reasons\'])\n    return calibration,speed_rows,records\n\n\nif __name__==\'__main__\':run()\n', 'benchmark_speed.py': '"""Label-independent single-pass estimators; reference speeds are used only by evaluation/tuning."""\nfrom collections import defaultdict\nfrom pathlib import Path\nimport json\nimport math\n\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\nimport calibrate_speed as geometry\n\n\ndef target_records(observations, report, min_height=0, max_height=10000, corner_order=\'checkpoint\'):\n    prepared=geometry.prepare(observations,report[\'ground_indices\'],report[\'width\'],report[\'height\'],corner_order)\n    boxes={(r[\'track_id\'],r[\'frame_index\']):r[\'baseline_box\'] for r in observations}\n    groups=defaultdict(list)\n    for row in prepared:\n        box=boxes[row[\'track_id\'],row[\'frame_index\']]\n        if min_height<=box[3]-box[1]<=max_height:groups[row[\'track_id\']].append(row)\n    if not groups:return []\n    # The moving pass with the largest observed footprint is the target, independent of speed labels.\n    target=max(groups.values(),key=lambda rows:max(abs(cv2.contourArea(np.array(r[\'quad\'],np.float32))) for r in rows))\n    return sorted(target,key=lambda r:r[\'time_s\'])\n\n\ndef estimate_single_pass(observations, report, config):\n    rows=target_records(observations,report,config.get(\'min_height\',0),config.get(\'max_height\',10000),config.get(\'corner_order\',\'checkpoint\'))\n    if len(rows)<8 or rows[-1][\'time_s\']-rows[0][\'time_s\']<.25:\n        return {\'available\':False,\'reason\':\'insufficient_unclipped_ground_geometry\',\'observations\':len(rows)}\n    length=config.get(\'length_m\',4.5);width=config.get(\'width_m\',1.8)\n    H,diagnostics=geometry.fit_mapping(rows,report[\'width\'],report[\'height\'],length,width,\n                                      legacy_normalization=config.get(\'legacy_normalization\',False))\n    calculated=geometry.estimate_track(rows,H)\n    result={\'available\':True,\'raw_kmh\':calculated[\'experimental_kmh\'],\n            \'estimated_kmh\':calculated[\'experimental_kmh\']*config.get(\'scale\',1),\n            \'time_interval_s\':[rows[0][\'time_s\'],rows[-1][\'time_s\']],\n            \'observations\':len(rows),\'track_id\':rows[0][\'track_id\'],\n            \'mapping\':H.tolist(),\'diagnostics\':diagnostics,\'motion_diagnostics\':calculated,\n            \'support_hull_image\':cv2.convexHull(np.array([r[\'quad\'] for r in rows],np.float32).reshape(-1,2)).reshape(-1,2).tolist(),\n            \'method\':\'single_pass_rectangle_calibration\',\'reference_speed_used\':False}\n    if config.get(\'method\')==\'projective_delay\':\n        delay=projective_delay(rows,length)\n        if not delay[\'available\']:return delay\n        result.update(delay)\n        result[\'estimated_kmh\']=result[\'raw_kmh\']*config.get(\'scale\',1)\n    result[\'calibration_length_m\']=length\n    result[\'scale_correction\']=config.get(\'scale\',1)\n    return result\n\n\ndef projective_delay(rows,length_m):\n    """Match front/rear ground positions through a shared 1-D projective motion curve.\n\n    For constant speed, the rear follows the front\'s path after a delay L/v.\n    Fitting this delay estimates speed without reading a reference speed or clip filename.\n    """\n    quads=np.array([r[\'quad\'] for r in rows]);centers=quads.mean(axis=1)\n    direction=centers[-1]-centers[0];direction/=np.linalg.norm(direction)\n    front=(quads[:,1]+quads[:,2])/2;rear=(quads[:,0]+quads[:,3])/2\n    if np.median((front-rear)@direction)<0:front,rear=rear,front\n    front=front@direction;rear=rear@direction\n    times=np.array([r[\'time_s\'] for r in rows]);span=times[-1]-times[0]\n    t=(times-times[0])/span\n    origin=np.mean(np.r_[front,rear]);normalizer=max(np.ptp(np.r_[front,rear]),1)\n    front=(front-origin)/normalizer;rear=(rear-origin)/normalizer\n    linear=np.polyfit(t,front,1)\n\n    def residual(parameters):\n        a,b,c,log_delay=parameters;delay=np.exp(log_delay)\n        def prediction(s):\n            den=1+c*s\n            return (a+b*s)/np.maximum(den,.02),np.maximum(.03-den,0)*10\n        f,fp=prediction(t);r,rp=prediction(t-delay)\n        return np.r_[f-front,r-rear,fp,rp]\n    fits=[]\n    for c in [-.8,-.4,0,.5,1.5]:\n        initial=[linear[1],linear[0],c,math.log(.25/span)]\n        fit=least_squares(residual,initial,bounds=([-5,-20,-.95,math.log(.03/span)],\n                                                 [5,20,4,math.log(4/span)]),loss=\'soft_l1\',f_scale=.02,max_nfev=500)\n        fits.append((np.mean(residual(fit.x)**2),fit))\n    error,fit=min(fits,key=lambda pair:pair[0]);delay_s=float(np.exp(fit.x[3])*span)\n    return {\'available\':True,\'raw_kmh\':length_m/delay_s*3.6,\'method\':\'projective_delay\',\n            \'projective_delay_s\':delay_s,\'projective_fit_rms_pixels\':float(np.sqrt(error)*normalizer),\n            \'projective_parameters\':fit.x.tolist()}\n\n\ndef load_clip(directory):\n    directory=Path(directory)\n    observations=json.loads((directory/\'observations.json\').read_text())\n    report=json.loads((directory/\'geometry_run_report.json\').read_text())\n    # Recompute corner-order voting from observations, including early pilot outputs.\n    points=np.array([r[\'keypoints\'] for r in observations])\n    if len(points):\n        vote=points[:,:4,1].mean(axis=1)>points[:,4:,1].mean(axis=1)\n        report[\'ground_indices\']=list(range(4)) if vote.mean()>=.5 else list(range(4,8))\n        report[\'ground_order_vote_agreement\']=float(max(vote.mean(),1-vote.mean()))\n    return observations,report\n', 'reference_calibration.py': '"""Reference-assisted road calibration and verified transfer between overlapping road views."""\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares,linprog\n\nimport calibrate_speed as geo\nfrom benchmark_speed import target_records\n\n\ndef fit_motion(world,times):\n    t=times-times.mean()\n    initial=np.r_[np.polyfit(t,world[:,0],1),np.polyfit(t,world[:,1],1)]\n    def residual(p):return np.r_[p[0]*t+p[1]-world[:,0],p[2]*t+p[3]-world[:,1]]\n    # Use the same constant-velocity least-squares estimator for fitting and\n    # evaluation. Switching estimators after calibration changes its scale.\n    return SimpleNamespace(x=initial),residual(initial)\n\n\ndef refine_metric_scale(estimates,references):\n    """Choose one physical distance-scale multiplier that minimizes worst pilot error."""\n    estimates=np.asarray(estimates,dtype=float);references=np.asarray(references,dtype=float)\n    A=np.vstack([np.c_[estimates,-np.ones(len(estimates))],np.c_[-estimates,-np.ones(len(estimates))]])\n    b=np.r_[references,-references]\n    result=linprog([0,1],A_ub=A,b_ub=b,bounds=[(.1,10),(0,None)],method=\'highs\')\n    if not result.success:raise ValueError(\'Pilot metric-scale refinement failed.\')\n    return float(result.x[0]),float(result.x[1])\n\n\ndef register_road(reference,current):\n    """Find the ground-plane transform using foreground road texture, not speed labels."""\n    reference=cv2.resize(reference,(960,540));current=cv2.resize(current,(960,540))\n    if reference.ndim==3:reference=cv2.cvtColor(reference,cv2.COLOR_BGR2GRAY)\n    if current.ndim==3:current=cv2.cvtColor(current,cv2.COLOR_BGR2GRAY)\n    mask=np.zeros((540,960),np.uint8);mask[324:,240:941]=255\n    detector=cv2.SIFT_create(nfeatures=8000,contrastThreshold=.01)\n    kr,dr=detector.detectAndCompute(reference,mask);kc,dc=detector.detectAndCompute(current,mask)\n    if dr is None or dc is None:return None,{\'accepted\':False,\'reason\':\'no_road_features\'}\n    pairs=cv2.BFMatcher().knnMatch(dc,dr,k=2)\n    matches=[a for pair in pairs if len(pair)==2 for a,b in [pair] if a.distance<.7*b.distance]\n    if len(matches)<12:return None,{\'accepted\':False,\'reason\':\'insufficient_road_matches\',\'matches\':len(matches)}\n    source=np.float32([kc[m.queryIdx].pt for m in matches]);dest=np.float32([kr[m.trainIdx].pt for m in matches])\n    H,inliers=cv2.findHomography(source,dest,cv2.RANSAC,2)\n    if H is None or inliers is None:return None,{\'accepted\':False,\'reason\':\'road_alignment_failed\'}\n    selected=inliers.ravel().astype(bool);count=int(selected.sum())\n    coverage=min(abs(cv2.contourArea(cv2.convexHull(p[selected])))/(960*540) for p in [source,dest]) if count>=3 else 0\n    predicted=geo.project(source[selected],H)\n    rms=float(np.sqrt(np.mean(np.sum((predicted-dest[selected])**2,axis=1))))\n    accepted=count>=12 and count/len(matches)>=.5 and coverage>=.015 and rms<=2\n    evidence={\'accepted\':bool(accepted),\'matches\':len(matches),\'inliers\':count,\'road_coverage\':coverage,\n              \'alignment_rms_pixels_at_960\':rms,\'reason\':\'verified_road_overlap\' if accepted else \'uncertain_road_alignment\'}\n    scale=np.diag([.5,.5,1])\n    return (np.linalg.inv(scale)@H@scale if accepted else None),evidence\n\n\ndef trajectory_points(observations,report):\n    rows=target_records(observations,report,80,500)\n    if len(rows)<8:return None,None,rows\n    quads=np.array([r[\'quad\'] for r in rows])\n    # Nearest face\'s ground midpoint avoids averaging unreliable hidden rear corners.\n    points=(quads[:,0]+quads[:,3])/2\n    times=np.array([r[\'time_s\'] for r in rows])\n    return points,times,rows\n\n\ndef fit_reference_mapping(clips):\n    """Fit one local projective metric mapping using designated pilot speeds only.\n\n    Each item has aligned points, times, quads and an explicitly supplied pilot reference.\n    Results on these same pilots are fitting diagnostics, not independent accuracy evidence.\n    """\n    records=[row for clip in clips for row in clip[\'rows\']]\n    quads=np.array([r[\'quad\'] for r in records]);center=np.median(quads.reshape(-1,2),axis=0)\n    N=np.array([[2/1920,0,-2*center[0]/1920],[0,2/1920,-2*center[1]/1920],[0,0,1]])\n    _,initial=geo.fit_mapping(records,1920,1080,4.9,1.87)\n\n    def mapping(parameters):\n        scale=np.exp(parameters[4])\n        return np.diag([scale,scale,1])@geo.compose(parameters[:4],N)\n\n    def residual(parameters):\n        H=mapping(parameters);out=[]\n        for clip in clips:\n            world=geo.project(clip[\'points\'],H);t=clip[\'times\']-clip[\'times\'].mean()\n            motion,_=fit_motion(world,clip[\'times\'])\n            velocity=motion.x[[0,2]]\n            prediction=world.mean(axis=0)+t[:,None]*velocity\n            target=clip[\'reference_kmh\']/3.6\n            out.extend(((world-prediction)/max(target,1)*3).ravel())\n            out.append((np.linalg.norm(velocity)-target)*50)\n        angles,parallel,lengths,widths=geo.measures(geo.project(quads,H))\n        # Predicted cuboids are approximate. Keep their shape as a weak regularizer.\n        out.extend((angles*.1).ravel())\n        out.extend((parallel*.1).ravel())\n        den=(np.c_[quads.reshape(-1,2),np.ones(len(quads)*4)]@H.T)[:,2]\n        out.extend(np.maximum(.03-den,0)*10)\n        return np.array(out)\n\n    fits=[]\n    for scale in [1,3,10,30]:\n        parameters=initial[\'theta\']+[np.log(scale)]\n        fit=least_squares(residual,parameters,bounds=([-100,-100,-7,-30,-5],[100,100,7,30,10]),\n                          loss=\'soft_l1\',f_scale=.1,max_nfev=1000)\n        fits.append((float(np.mean(residual(fit.x)**2)),fit))\n    error,fit=min(fits,key=lambda item:item[0]);H=mapping(fit.x)\n    raw=[]\n    for clip in clips:\n        motion,_=fit_motion(geo.project(clip[\'points\'],H),clip[\'times\'])\n        raw.append(float(np.hypot(motion.x[0],motion.x[2])*3.6))\n    scale,max_error=refine_metric_scale(raw,[c[\'reference_kmh\'] for c in clips])\n    H=np.diag([scale,scale,1])@H\n    support=cv2.convexHull(np.concatenate([c[\'points\'] for c in clips]).astype(np.float32)).reshape(-1,2)\n    return {\'H_image_to_meters\':H.tolist(),\'support_hull_image\':support.tolist(),\n            \'fit_rms\':float(np.sqrt(error)),\'parameters\':fit.x.tolist(),\n            \'reference_videos\':[c[\'video\'] for c in clips],\n            \'reference_speeds_kmh\':[c[\'reference_kmh\'] for c in clips],\n            \'metric_scale_refinement\':{\'multiplier\':scale,\'objective\':\'minimum worst absolute pilot speed error\',\n                                       \'maximum_pilot_error_kmh\':max_error,\'unrefined_pilot_estimates_kmh\':raw},\n            \'method\':\'reference_assisted_local_projective_calibration\',\n            \'physical_accuracy_verified\':False,\'scope\':\'Registered road region; this vehicle and recording setup.\'}\n\n\ndef estimate_reference_speed(points,times,mapping):\n    world=geo.project(points,np.array(mapping[\'H_image_to_meters\']))\n    fit,residual=fit_motion(world,times)\n    return {\'estimated_kmh\':float(np.hypot(fit.x[0],fit.x[2])*3.6),\n            \'trajectory_rms_m\':float(np.sqrt(np.mean(residual**2))),\n            \'time_interval_s\':[float(times[0]),float(times[-1])],\n            \'estimator_reads_reference_label\':False}\n', 'calibration_candidates.py': '"""Independent per-video calibration candidates; prediction functions accept no speed labels.\n\nThe temporal and rigid-bundle methods are new implementations of projective and\nrigid-body geometry, not reproductions of any authors\' trained keypoint system.\n"""\nimport math\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\nfrom benchmark_speed import target_records, projective_delay\n\n\ndef select_rows(observations, report, minimum=100, maximum=400):\n    rows = target_records(observations, report, minimum, maximum)\n    original={(r[\'track_id\'],r[\'frame_index\']):r for r in observations}\n    rows=[{**original[r[\'track_id\'],r[\'frame_index\']],**r} for r in rows]\n    for row in rows:row[\'box_height\']=row[\'baseline_box\'][3]-row[\'baseline_box\'][1]\n    width, height = report[\'width\'], report[\'height\']\n    rows = [r for r in rows if np.all(np.asarray(r[\'quad\'])[:, 0] > width*.02)\n            and np.all(np.asarray(r[\'quad\'])[:, 0] < width*.98)\n            and np.all(np.asarray(r[\'quad\'])[:, 1] < height*.98)]\n    return sorted(rows, key=lambda r:r[\'time_s\'])\n\n\ndef temporal_delay_2d(rows, length_m=4.9):\n    """Fit front/rear ground midpoints to one 2D rational constant-motion curve."""\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_geometry\'}\n    q=np.asarray([r[\'quad\'] for r in rows]);times=np.asarray([r[\'time_s\'] for r in rows])\n    span=float(np.ptp(times))\n    if span<.2:return {\'available\':False,\'reason\':\'short_interval\'}\n    front=(q[:,0]+q[:,3])/2;rear=(q[:,1]+q[:,2])/2\n    direction=np.mean(q[-3:],axis=(0,1))-np.mean(q[:3],axis=(0,1))\n    if np.median((front-rear)@direction)<0:front,rear=rear,front\n    origin=np.mean(np.r_[front,rear],axis=0);scale=max(np.ptp(np.r_[front,rear],axis=0).max(),1)\n    f=(front-origin)/scale;r=(rear-origin)/scale;t=(times-times[0])/span\n    linear=np.array([np.polyfit(t,f[:,axis],1) for axis in [0,1]])\n    def curve(p,s):return (p[:2]+s[:,None]*p[2:4])/np.maximum(1+p[4]*s[:,None],.01)\n    def residual(p):\n        d=np.exp(p[5]);den=np.r_[1+p[4]*t,1+p[4]*(t-d)]\n        return np.r_[(curve(p,t)-f).ravel(),(curve(p,t-d)-r).ravel(),np.maximum(.03-den,0)*10]\n    fits=[]\n    for perspective in [-.7,-.3,0,.7,2]:\n        initial=np.r_[linear[:,1],linear[:,0],perspective,np.log(.25/span)]\n        fit=least_squares(residual,initial,bounds=([-5,-5,-20,-20,-.95,np.log(.03/span)],\n            [5,5,20,20,4,np.log(4/span)]),loss=\'soft_l1\',f_scale=.015,max_nfev=500)\n        error=np.mean(residual(fit.x)**2);fits.append((error,fit))\n    error,fit=min(fits,key=lambda x:x[0]);delay=float(np.exp(fit.x[5])*span)\n    return {\'available\':True,\'estimated_kmh\':length_m/delay*3.6,\'method\':\'temporal_projective_2d\',\n        \'delay_s\':delay,\'time_interval_s\':[float(times[0]),float(times[-1])],\n        \'track_id\':rows[0][\'track_id\'],\'observations\':len(rows),\n        \'reprojection_rms_pixels\':float(np.sqrt(error)*scale),\n        \'parameters\':fit.x.tolist(),\'curve_origin\':origin.tolist(),\'curve_scale\':float(scale),\n        \'fitted_front_image\':(curve(fit.x,t)*scale+origin).tolist(),\n        \'fitted_rear_image\':(curve(fit.x,t-np.exp(fit.x[5]))*scale+origin).tolist(),\n        \'curve_times_s\':times.tolist(),\n        \'support_hull_image\':cv2.convexHull(q.reshape(-1,2).astype(np.float32)).reshape(-1,2).tolist(),\n        \'length_prior_m\':length_m,\'reference_label_used\':False}\n\n\ndef corner_vanishing_point(rows):\n    """Robust intersection of tracked longitudinal-corner motion lines."""\n    quads=np.asarray([r[\'quad\'] for r in rows]);lines=[]\n    for index in range(4):\n        vx,vy,x,y=cv2.fitLine(quads[:,index].astype(np.float32),cv2.DIST_HUBER,0,.01,.01).ravel()\n        lines.append([-float(vy),float(vx),float(vy*x-vx*y)])\n    lines=np.array(lines);initial=np.linalg.lstsq(lines[:,:2],-lines[:,2],rcond=None)[0]\n    fit=least_squares(lambda p:lines[:,:2]@p+lines[:,2],initial,loss=\'soft_l1\',f_scale=3)\n    return fit.x,float(np.sqrt(np.mean((lines[:,:2]@fit.x+lines[:,2])**2)))\n\n\ndef shared_vp_delay(rows,length_m=4.9,vanishing_point=None,vp_weight=.02):\n    """Fit both longitudinal sides with a shared motion vanishing point and delay.\n\n    Each side retains its own projective depth. This avoids averaging projected\n    corners before accounting for perspective. An optional flow VP is speed-blind.\n    """\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_geometry\'}\n    q=np.asarray([r[\'quad\'] for r in rows]);times=np.array([r[\'time_s\'] for r in rows])\n    span=float(np.ptp(times));t=(times-times[0])/span\n    fronts=q[:,[0,3]];rears=q[:,[1,2]]\n    direction=q[-1].mean(axis=0)-q[0].mean(axis=0)\n    if np.median(np.sum((fronts-rears)*direction,axis=2))<0:fronts,rears=rears,fronts\n    origin=q.mean(axis=(0,1));scale=max(np.ptp(q.reshape(-1,2),axis=0).max(),1)\n    front=(fronts-origin)/scale;rear=(rears-origin)/scale\n    inferred,uncertainty=corner_vanishing_point(rows)\n    if vanishing_point is None:vanishing_point=inferred\n    vp=(np.asarray(vanishing_point)-origin)/scale\n    # Allow small VP refinement for noisy correspondences while retaining the geometric prior.\n    sigma=max(uncertainty*4,15)/scale\n    def curve(p,s):\n        current_vp=p[7:9];A=p[:4].reshape(2,2);c=p[4:6]\n        return current_vp[None,None]+A[None]/np.maximum(1+s[:,None,None]*c[None,:,None],.02)\n    def residual(p):\n        d=np.exp(p[6]);den=np.r_[(1+t[:,None]*p[4:6]).ravel(),(1+(t[:,None]-d)*p[4:6]).ravel()]\n        return np.r_[(curve(p,t)-front).ravel(),(curve(p,t-d)-rear).ravel(),\n                     np.maximum(.03-den,0)*10,(p[7:9]-vp)/sigma*vp_weight]\n    fits=[]\n    for c in [-.75,-.4,0,.5,2]:\n        initial=np.r_[(front[0]-vp).ravel(),c,c,np.log(.25/span),vp]\n        lower=[-10]*4+[-.95,-.95,np.log(.03/span),-10,-10]\n        upper=[10]*4+[4,4,np.log(4/span),10,10]\n        initial=np.clip(initial,np.array(lower)+1e-6,np.array(upper)-1e-6)\n        fit=least_squares(residual,initial,bounds=(lower,upper),loss=\'soft_l1\',f_scale=.015,max_nfev=500)\n        score=float(np.mean(residual(fit.x)**2));fits.append((score,fit))\n    score,fit=min(fits,key=lambda x:x[0]);delay=float(np.exp(fit.x[6])*span)\n    return {\'available\':True,\'estimated_kmh\':length_m/delay*3.6,\'method\':\'shared_vanishing_point_temporal_calibration\',\n        \'track_id\':rows[0][\'track_id\'],\'observations\':len(rows),\'delay_s\':delay,\'length_prior_m\':length_m,\n        \'time_interval_s\':[float(times[0]),float(times[-1])],\'parameters\':fit.x.tolist(),\n        \'vanishing_point_image\':(fit.x[7:9]*scale+origin).tolist(),\n        \'initial_vanishing_point_image\':np.asarray(vanishing_point).tolist(),\n        \'corner_line_rms_pixels\':uncertainty,\'reprojection_rms_pixels\':float(np.sqrt(score)*scale),\n        \'fitted_front_image\':(curve(fit.x,t)*scale+origin).tolist(),\n        \'fitted_rear_image\':(curve(fit.x,t-np.exp(fit.x[6]))*scale+origin).tolist(),\n        \'curve_times_s\':times.tolist(),\n        \'curve_origin\':origin.tolist(),\'curve_scale\':float(scale),\'vanishing_point_weight\':vp_weight,\n        \'reference_label_used\':False}\n\n\ndef rigid_cuboid_bundle(observations,report,length_m=4.9,width_m=1.87,height_m=1.45,distortion=False):\n    """Jointly fit one camera and constant rigid 3D motion to many cuboid observations.\n\n    Camera principal point is assumed at image centre; focal length and rotation\n    are fitted. Object dimensions are priors, not independently measured values.\n    """\n    rows=select_rows(observations,report,100,400)\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_geometry\'}\n    # Bound computation and balance temporal support without reference-speed input.\n    rows=[rows[i] for i in np.unique(np.linspace(0,len(rows)-1,min(36,len(rows))).astype(int))]\n    times=np.array([r[\'time_s\'] for r in rows]);dt=times-times.mean()\n    base=np.array([[-width_m/2,0,0],[-width_m/2,0,length_m],\n                   [width_m/2,0,length_m],[width_m/2,0,0]],float)\n    object_points=np.r_[base,base+[0,-height_m,0]]\n    indices=report[\'ground_indices\']+[i for i in range(8) if i not in report[\'ground_indices\']]\n    points=np.array([r[\'keypoints\'] for r in rows])[:,indices,:2]\n    pixel_scale=np.maximum(np.array([r[\'box_height\'] for r in rows]),100)\n    center=np.median(points[:,:4].mean(axis=1),axis=0);cx=report[\'width\']/2;cy=report[\'height\']/2\n    def camera(p):\n        R=cv2.Rodrigues(p[:3])[0];T=np.r_[p[3:5],np.exp(p[5])]\n        v=np.exp(p[6]);f=np.exp(p[7]);k=p[8] if distortion else 0\n        XYZ=object_points[None]@R.T+T-dt[:,None,None]*v*R[:,2]\n        xy=XYZ[:,:,:2]/np.maximum(XYZ[:,:,2:],.1)\n        xy*=1+k*np.sum(xy**2,axis=2,keepdims=True)\n        image=xy*f+[cx,cy]\n        return image,XYZ,R,T,f\n    def residual(p):\n        image,XYZ,*_=camera(p)\n        return np.r_[((image-points)/pixel_scale[:,None,None]*200).ravel(),np.maximum(1-XYZ[:,:,2],0).ravel()*20]\n    lower=[-.8,-1.5,-.4,-100,-10,np.log(3),np.log(.5),np.log(250)]\n    upper=[.8,1.5,.4,100,10,np.log(1000),np.log(70),np.log(4000)]\n    if distortion:lower+=[-.5];upper+=[.5]\n    fits=[]\n    for focal,yaw in [(600,0),(1000,0),(1600,0),(1000,.35),(1000,-.35)]:\n        depth=20;initial=[0,yaw,0,(center[0]-cx)/focal*depth,(center[1]-cy)/focal*depth,\n                         np.log(depth),np.log(12),np.log(focal)]\n        if distortion:initial+=[0]\n        initial=np.clip(initial,np.array(lower)+1e-6,np.array(upper)-1e-6)\n        fit=least_squares(residual,initial,bounds=(lower,upper),loss=\'soft_l1\',f_scale=3,max_nfev=300)\n        score=float(np.mean(np.minimum(residual(fit.x)**2,100**2)));fits.append((score,fit))\n    score,fit=min(fits,key=lambda x:x[0]);image,XYZ,R,T,f=camera(fit.x)\n    K=np.array([[f,0,cx],[0,f,cy],[0,0,1]])\n    H=np.linalg.inv(K@np.c_[R[:,0],R[:,2],T])\n    return {\'available\':bool(np.min(XYZ[:,:,2])>0),\'estimated_kmh\':float(np.exp(fit.x[6])*3.6),\n        \'method\':\'rigid_cuboid_bundle_distortion\' if distortion else \'rigid_cuboid_bundle\',\n        \'parameters\':fit.x.tolist(),\'focal_length_pixels\':float(f),\'radial_k1\':float(fit.x[8]) if distortion else 0,\n        \'mapping\':H.tolist(),\'support_hull_image\':cv2.convexHull(points[:,:4].reshape(-1,2).astype(np.float32)).reshape(-1,2).tolist(),\n        \'time_interval_s\':[float(times[0]),float(times[-1])],\'track_id\':rows[0][\'track_id\'],\n        \'observations\':len(rows),\'weighted_reprojection_rms\':float(np.sqrt(score)),\n        \'dimensions_m\':[length_m,width_m,height_m],\'reference_label_used\':False}\n\n\ndef silhouette_bundle(boxes,times,width,height,shape=\'sedan\',length_m=4.9,width_m=1.87,height_m=1.45):\n    """Fit a rigid constant-motion vehicle envelope directly to detector boxes.\n\n    This avoids inferred hidden cuboid corners. The generic envelope is a size\n    prior, not an identified CAD model of the Mercedes.\n    """\n    boxes=np.asarray(boxes,float);times=np.asarray(times,float)\n    keep=(boxes[:,3]-boxes[:,1]>=70)&(boxes[:,3]-boxes[:,1]<=450)&(boxes[:,0]>width*.02)&(boxes[:,2]<width*.98)\n    boxes=boxes[keep];times=times[keep]\n    if len(boxes)<8:return {\'available\':False,\'reason\':\'insufficient_unclipped_boxes\'}\n    selected=np.unique(np.linspace(0,len(boxes)-1,min(60,len(boxes))).astype(int))\n    boxes=boxes[selected];times=times[selected];dt=times-times.mean()\n    if shape==\'cuboid\':\n        vertices=np.array([[x,y,z] for x in [-width_m/2,width_m/2]\n                           for y in [0,-height_m] for z in [0,length_m]])\n    else:\n        # Front/rear bumper, bonnet, windscreen, roof and rear deck sections.\n        sections=[(0,.85,1),(.22,1.0,1),(1.25,1.0,.98),(1.8,height_m,.77),\n                  (3.2,height_m,.77),(3.85,1.0,.96),(length_m,.83,1)]\n        vertices=np.array([[sign*width_m/2*w,-h,z] for z,h,w in sections for sign in [-1,1]]+\n                          [[sign*width_m/2,0,z] for z in [.85,length_m-.85] for sign in [-1,1]])\n    cx=width/2;cy=height/2;center=np.median((boxes[:,:2]+boxes[:,2:])/2,axis=0)\n    pixel_scale=np.maximum(boxes[:,3]-boxes[:,1],70)\n    def camera(p):\n        R=cv2.Rodrigues(p[:3])[0];T=np.r_[p[3:5],np.exp(p[5])];v=np.exp(p[6]);f=np.exp(p[7])\n        XYZ=vertices[None]@R.T+T-dt[:,None,None]*v*R[:,2]\n        xy=XYZ[:,:,:2]/np.maximum(XYZ[:,:,2:],.1)*f+[cx,cy]\n        predicted=np.c_[xy.min(axis=1),xy.max(axis=1)]\n        return predicted,XYZ,R,T,f\n    def residual(p):\n        predicted,XYZ,*_=camera(p)\n        return np.r_[((predicted-boxes)/pixel_scale[:,None]*200).ravel(),np.maximum(1-XYZ[:,:,2],0).ravel()*20]\n    lower=[-.6,-1.5,-.2,-100,-10,np.log(3),np.log(.5),np.log(250)]\n    upper=[.6,1.5,.2,100,10,np.log(1000),np.log(70),np.log(4000)]\n    fits=[]\n    for focal,yaw in [(600,0),(1000,0),(1600,0),(1000,.4),(1000,-.4)]:\n        initial=[0,yaw,0,(center[0]-cx)/focal*20,(center[1]-cy)/focal*20,\n                 np.log(20),np.log(12),np.log(focal)]\n        fit=least_squares(residual,initial,bounds=(lower,upper),loss=\'soft_l1\',f_scale=2,max_nfev=500)\n        score=float(np.mean(np.minimum(residual(fit.x)**2,10000)));fits.append((score,fit))\n    score,fit=min(fits,key=lambda x:x[0]);predicted,XYZ,R,T,f=camera(fit.x)\n    K=np.array([[f,0,cx],[0,f,cy],[0,0,1]])\n    H=np.linalg.inv(K@np.c_[R[:,0],R[:,2],T])\n    anchors=np.c_[(boxes[:,0]+boxes[:,2])/2,boxes[:,3]]\n    return {\'available\':bool(np.min(XYZ[:,:,2])>0),\'estimated_kmh\':float(np.exp(fit.x[6])*3.6),\n        \'method\':\'silhouette_bundle_\'+shape,\'parameters\':fit.x.tolist(),\'mapping\':H.tolist(),\n        \'focal_length_pixels\':float(f),\'weighted_reprojection_rms\':float(np.sqrt(score)),\n        \'time_interval_s\':[float(times[0]),float(times[-1])],\'observations\':len(boxes),\n        \'fitted_boxes\':predicted.tolist(),\'observed_boxes\':boxes.tolist(),\'times_s\':times.tolist(),\n        \'support_hull_image\':cv2.convexHull(anchors.astype(np.float32)).reshape(-1,2).tolist(),\n        \'dimensions_m\':[length_m,width_m,height_m],\'reference_label_used\':False}\n\n\ndef vp_rigid_bundle(observations,report,vanishing_point,length_m=4.9,width_m=1.87,height_m=1.45,fit_length=False):\n    """Rigid camera fit whose translation axis is fixed by visual optical flow."""\n    rows=select_rows(observations,report)\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_geometry\'}\n    rows=[rows[i] for i in np.unique(np.linspace(0,len(rows)-1,min(36,len(rows))).astype(int))]\n    times=np.array([r[\'time_s\'] for r in rows]);dt=times-times.mean()\n    base=np.array([[-width_m/2,0,0],[-width_m/2,0,length_m],[width_m/2,0,length_m],[width_m/2,0,0]])\n    vertices=np.r_[base,base+[0,-height_m,0]]\n    indices=report[\'ground_indices\']+[i for i in range(8) if i not in report[\'ground_indices\']]\n    observed=np.array([r[\'keypoints\'] for r in rows])[:,indices,:2]\n    pixel_scale=np.array([r[\'box_height\'] for r in rows]);cx=report[\'width\']/2;cy=report[\'height\']/2\n    vp=np.asarray(vanishing_point);center=np.median(observed[:,:4].mean(axis=1),axis=0)\n    def camera(p):\n        roll,tx,ty,logz,logv,logf=p[:6];f=np.exp(logf)\n        z=np.r_[(vp-[cx,cy])/f,1];z/=np.linalg.norm(z)\n        x=np.cross([0,1,0],z);x/=np.linalg.norm(x);y=np.cross(z,x)\n        R=np.c_[x*np.cos(roll)+y*np.sin(roll),-x*np.sin(roll)+y*np.cos(roll),z]\n        T=np.array([tx,ty,np.exp(logz)]);v=np.exp(logv)\n        current=vertices.copy()\n        if fit_length:current[:,2]*=np.exp(p[6])/length_m\n        XYZ=current[None]@R.T+T-dt[:,None,None]*v*z\n        xy=XYZ[:,:,:2]/np.maximum(XYZ[:,:,2:],.1)*f+[cx,cy]\n        return xy,XYZ,R,T,f\n    def residual(p):\n        xy,XYZ,*_=camera(p)\n        values=np.r_[((xy-observed)/pixel_scale[:,None,None]*200).ravel(),np.maximum(1-XYZ[:,:,2],0).ravel()*20]\n        return np.r_[values,(p[6]-np.log(length_m))] if fit_length else values\n    lower=[-.4,-100,-10,np.log(3),np.log(.5),np.log(250)]\n    upper=[.4,100,10,np.log(1000),np.log(70),np.log(4000)]\n    if fit_length:lower+=[np.log(2.5)];upper+=[np.log(7.5)]\n    fits=[]\n    for f in [600,1000,1600]:\n        initial=[0,(center[0]-cx)/f*20,(center[1]-cy)/f*20,np.log(20),np.log(12),np.log(f)]\n        if fit_length:initial+=[np.log(length_m)]\n        fit=least_squares(residual,initial,bounds=(lower,upper),loss=\'soft_l1\',f_scale=3,max_nfev=400)\n        score=float(np.mean(np.minimum(residual(fit.x)**2,10000)));fits.append((score,fit))\n    score,fit=min(fits,key=lambda x:x[0]);xy,XYZ,R,T,f=camera(fit.x)\n    K=np.array([[f,0,cx],[0,f,cy],[0,0,1]]);H=np.linalg.inv(K@np.c_[R[:,0],R[:,2],T])\n    return {\'available\':bool(np.min(XYZ[:,:,2])>0),\'estimated_kmh\':float(np.exp(fit.x[4])*3.6),\n        \'method\':\'optical_flow_vp_rigid_bundle\',\'mapping\':H.tolist(),\'parameters\':fit.x.tolist(),\n        \'focal_length_pixels\':float(f),\'vanishing_point_image\':vp.tolist(),\n        \'track_id\':rows[0][\'track_id\'],\'time_interval_s\':[float(times[0]),float(times[-1])],\n        \'observations\':len(rows),\'weighted_reprojection_rms\':float(np.sqrt(score)),\n        \'support_hull_image\':cv2.convexHull(observed[:,:4].reshape(-1,2).astype(np.float32)).reshape(-1,2).tolist(),\n        \'reference_label_used\':False,\'dimensions_m\':[float(np.exp(fit.x[6])) if fit_length else length_m,width_m,height_m],\n        \'effective_cuboid_length_fitted\':fit_length}\n', 'flow_calibration.py': '"""Forward/backward checked optical flow and robust contact-anchor transport.\n\nInspired by dynamic vehicle-template flow, with the metric ground map supplied\nby a separate calibration method. This is not the 36-keypoint paper\'s model.\n"""\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom benchmark_speed import target_records\nfrom calibrate_speed import project\n\n\ndef extract_flow(video,observations,report,tracks):\n    rows=target_records(observations,report)\n    if not rows:return {\'available\':False,\'measurements\':[]}\n    tid=rows[0][\'track_id\'];by_frame={r[\'frame_index\']:r for r in tracks if r[\'track_id\']==tid}\n    cap=cv2.VideoCapture(str(video));assert cap.isOpened()\n    ok,previous=cap.read();index=0;measurements=[]\n    try:\n        while ok:\n            ok,current=cap.read()\n            if not ok:break\n            a=by_frame.get(index);b=by_frame.get(index+1)\n            if a is not None and b is not None:\n                box=np.array(a[\'box\']);next_box=np.array(b[\'box\']);height=box[3]-box[1]\n                if 80<=height<=450 and min(box[0],report[\'width\']-box[2])>report[\'width\']*.02:\n                    union=np.r_[np.minimum(box[:2],next_box[:2])-20,np.maximum(box[2:],next_box[2:])+20]\n                    x0,y0,x1,y1=np.rint(union).astype(int)\n                    x0=max(x0,0);y0=max(y0,0);x1=min(x1,report[\'width\']);y1=min(y1,report[\'height\'])\n                    g0=cv2.cvtColor(previous[y0:y1,x0:x1],cv2.COLOR_BGR2GRAY)\n                    g1=cv2.cvtColor(current[y0:y1,x0:x1],cv2.COLOR_BGR2GRAY)\n                    mask=np.zeros_like(g0);offset=np.array([x0,y0])\n                    p0=(box[:2]+(box[2:]-box[:2])*.12-offset).astype(int)\n                    p1=(box[2:]-(box[2:]-box[:2])*.12-offset).astype(int)\n                    cv2.rectangle(mask,tuple(p0),tuple(p1),255,-1)\n                    features=cv2.goodFeaturesToTrack(g0,120,.01,4,mask=mask)\n                    if features is not None and len(features)>=8:\n                        forward,status,_=cv2.calcOpticalFlowPyrLK(g0,g1,features,None,winSize=(31,31),maxLevel=3)\n                        backward,bs,_=cv2.calcOpticalFlowPyrLK(g1,g0,forward,None,winSize=(31,31),maxLevel=3)\n                        old=features[:,0];new=forward[:,0]\n                        keep=(status[:,0]>0)&(bs[:,0]>0)&(np.linalg.norm(backward[:,0]-old,axis=1)<1)\n                        keep&=(np.linalg.norm(new-old,axis=1)>.25)\n                        if keep.sum()>=6:\n                            affine,inliers=cv2.estimateAffinePartial2D(old[keep],new[keep],method=cv2.RANSAC,ransacReprojThreshold=1.5,maxIters=1000)\n                            if affine is not None and inliers.sum()>=6:\n                                anchor=np.array([(box[0]+box[2])/2,box[3]])\n                                transported=affine@np.r_[anchor-offset,1]+offset\n                                selected=inliers[:,0].astype(bool)\n                                error=np.linalg.norm(old[keep][selected]@affine[:,:2].T+affine[:,2]-new[keep][selected],axis=1)\n                                measurements.append({\'frame_index\':index,\'time_s\':a[\'time_s\'],\n                                    \'dt_s\':b[\'time_s\']-a[\'time_s\'],\'anchor\':anchor.tolist(),\n                                    \'transported_anchor\':transported.tolist(),\'affine\':affine.tolist(),\n                                    \'crop_origin\':offset.tolist(),\'inliers\':int(inliers.sum()),\n                                    \'flow_rms_pixels\':float(np.sqrt(np.mean(error**2))),\n                                    \'box_height\':float(height)})\n            previous=current;index+=1\n    finally:cap.release()\n    return {\'available\':len(measurements)>=8,\'track_id\':tid,\'measurements\':measurements,\n            \'reference_label_used\':False,\'frames_decoded\':index+1}\n\n\ndef estimate_flow(features,mapping,trim_fraction=.1):\n    rows=features.get(\'measurements\',[])\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_checked_flow\'}\n    times=np.array([r[\'time_s\'] for r in rows]);lo,hi=np.quantile(times,[trim_fraction,1-trim_fraction])\n    rows=[r for r in rows if lo<=r[\'time_s\']<=hi]\n    points=np.array([r[\'anchor\'] for r in rows]);next_points=np.array([r[\'transported_anchor\'] for r in rows])\n    dt=np.array([r[\'dt_s\'] for r in rows])\n    velocities=(project(next_points,np.array(mapping))-project(points,np.array(mapping)))/dt[:,None]\n    speeds=np.linalg.norm(velocities,axis=1)*3.6\n    finite=np.isfinite(speeds)&(speeds>0)&(speeds<300)\n    if finite.sum()<8:return {\'available\':False,\'reason\':\'invalid_mapped_flow\'}\n    median=np.median(speeds[finite]);mad=np.median(np.abs(speeds[finite]-median))\n    keep=finite&(np.abs(speeds-median)<=max(4*1.4826*mad,2))\n    return {\'available\':True,\'estimated_kmh\':float(np.median(speeds[keep])),\n        \'method\':\'learned_map_and_optical_flow\',\'mapping\':np.asarray(mapping).tolist(),\n        \'track_id\':features[\'track_id\'],\'observations\':int(keep.sum()),\n        \'time_interval_s\':[float(times.min()),float(times.max())],\n        \'support_hull_image\':cv2.convexHull(points.astype(np.float32)).reshape(-1,2).tolist(),\n        \'per_frame_kmh\':speeds.tolist(),\'per_frame_keep\':keep.tolist(),\n        \'per_frame_times_s\':[r[\'time_s\'] for r in rows],\n        \'speed_mad_kmh\':float(mad),\'time_trim_fraction\':trim_fraction,\'reference_label_used\':False}\n', 'refine_flow.py': '"""Adaptive optical-flow intervals and speed-blind constant-motion map refinement."""\nfrom collections import deque\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\nfrom benchmark_speed import target_records\nfrom calibrate_speed import project\nfrom flow_calibration import estimate_flow\n\n\ndef extract_adaptive_flow(video,observations,report,tracks,max_gap=4,min_displacement=4):\n    rows=target_records(observations,report)\n    if not rows:return {\'available\':False,\'measurements\':[]}\n    tid=rows[0][\'track_id\'];by_frame={r[\'frame_index\']:r for r in tracks if r[\'track_id\']==tid}\n    cap=cv2.VideoCapture(str(video));assert cap.isOpened();buffer=deque(maxlen=max_gap+1)\n    measurements=[];index=0\n    def anchor(box):return np.array([(box[0]+box[2])/2,box[3]])\n    try:\n        while True:\n            ok,frame=cap.read()\n            if not ok:break\n            buffer.append((index,frame))\n            b=by_frame.get(index)\n            if b is not None:\n                for gap in range(1,min(max_gap,index)+1):\n                    a=by_frame.get(index-gap)\n                    if a is None:continue\n                    if np.linalg.norm(anchor(a[\'box\'])-anchor(b[\'box\']))>=min_displacement or gap==max_gap:\n                        break\n                else:a=None\n                if a is not None:\n                    old_index=index-gap;previous=next(im for number,im in buffer if number==old_index)\n                    box=np.array(a[\'box\']);next_box=np.array(b[\'box\']);height=box[3]-box[1]\n                    if 80<=height<=450 and min(box[0],report[\'width\']-box[2])>report[\'width\']*.02:\n                        union=np.r_[np.minimum(box[:2],next_box[:2])-20,np.maximum(box[2:],next_box[2:])+20]\n                        x0,y0,x1,y1=np.rint(union).astype(int);x0=max(x0,0);y0=max(y0,0)\n                        x1=min(x1,report[\'width\']);y1=min(y1,report[\'height\']);offset=np.array([x0,y0])\n                        g0=cv2.cvtColor(previous[y0:y1,x0:x1],cv2.COLOR_BGR2GRAY)\n                        g1=cv2.cvtColor(frame[y0:y1,x0:x1],cv2.COLOR_BGR2GRAY)\n                        mask=np.zeros_like(g0)\n                        cv2.rectangle(mask,tuple((box[:2]+(box[2:]-box[:2])*.12-offset).astype(int)),\n                            tuple((box[2:]-(box[2:]-box[:2])*.12-offset).astype(int)),255,-1)\n                        features=cv2.goodFeaturesToTrack(g0,120,.01,4,mask=mask)\n                        if features is not None and len(features)>=8:\n                            forward,status,_=cv2.calcOpticalFlowPyrLK(g0,g1,features,None,winSize=(31,31),maxLevel=3)\n                            backward,bs,_=cv2.calcOpticalFlowPyrLK(g1,g0,forward,None,winSize=(31,31),maxLevel=3)\n                            old=features[:,0];new=forward[:,0]\n                            keep=(status[:,0]>0)&(bs[:,0]>0)&(np.linalg.norm(backward[:,0]-old,axis=1)<1)\n                            keep&=(np.linalg.norm(new-old,axis=1)>.25)\n                            if keep.sum()>=6:\n                                affine,inliers=cv2.estimateAffinePartial2D(old[keep],new[keep],method=cv2.RANSAC,ransacReprojThreshold=1.5,maxIters=1000)\n                                if affine is not None and inliers.sum()>=6:\n                                    point=anchor(box);transported=affine@np.r_[point-offset,1]+offset\n                                    valid=inliers[:,0].astype(bool)\n                                    error=np.linalg.norm(old[keep][valid]@affine[:,:2].T+affine[:,2]-new[keep][valid],axis=1)\n                                    measurements.append({\'frame_index\':old_index,\'next_frame_index\':index,\n                                        \'frame_gap\':gap,\'time_s\':a[\'time_s\'],\'dt_s\':b[\'time_s\']-a[\'time_s\'],\n                                        \'anchor\':point.tolist(),\'transported_anchor\':transported.tolist(),\n                                        \'affine\':affine.tolist(),\'crop_origin\':offset.tolist(),\n                                        \'matched_points_image\':(old[keep][valid]+offset).tolist(),\n                                        \'matched_next_points_image\':(new[keep][valid]+offset).tolist(),\n                                        \'inliers\':int(inliers.sum()),\'flow_rms_pixels\':float(np.sqrt(np.mean(error**2))),\n                                        \'box_height\':float(height)})\n            index+=1\n    finally:cap.release()\n    return {\'available\':len(measurements)>=8,\'track_id\':tid,\'measurements\':measurements,\n            \'frames_decoded\':index,\'maximum_frame_gap\':max_gap,\'minimum_target_pixel_displacement\':min_displacement,\n            \'reference_label_used\':False}\n\n\ndef flow_vanishing_point(features):\n    points=[];weights=[]\n    for row in features.get(\'measurements\',[]):\n        A=np.asarray(row[\'affine\']);C=np.eye(2)-A[:,:2]\n        if abs(np.linalg.det(C))<1e-6:continue\n        center=np.linalg.solve(C,A[:,2]+C@np.asarray(row[\'crop_origin\']))\n        if not np.isfinite(center).all() or abs(center).max()>10000:continue\n        points.append(center);weights.append(row[\'inliers\'])\n    if len(points)<8:return {\'available\':False,\'reason\':\'insufficient_expansion_centres\'}\n    points=np.asarray(points);initial=np.median(points,axis=0)\n    mad=np.median(np.abs(points-initial),axis=0)\n    keep=np.all(np.abs(points-initial)<=np.maximum(4*1.4826*mad,15),axis=1)\n    point=np.median(points[keep],axis=0)\n    return {\'available\':True,\'point\':point.tolist(),\'observations\':int(keep.sum()),\n            \'median_absolute_deviation_pixels\':np.median(np.abs(points[keep]-point),axis=0).tolist(),\n            \'reference_label_used\':False}\n\n\ndef refine_constant_motion(features,mapping):\n    """Constrain mapped flow to one velocity without knowing its magnitude.\n\n    Retain the learned map\'s Jacobian at the centre of observed support to fix\n    metric scale. This assumption is specific to these constant-speed passes.\n    """\n    rows=features.get(\'measurements\',[])\n    if len(rows)<12:return {\'available\':False,\'reason\':\'insufficient_checked_flow\'}\n    times=np.array([r[\'time_s\'] for r in rows]);lo,hi=np.quantile(times,[.1,.9])\n    rows=[r for r in rows if lo<=r[\'time_s\']<=hi]\n    points=np.array([r[\'anchor\'] for r in rows]);next_points=np.array([r[\'transported_anchor\'] for r in rows])\n    dt=np.array([r[\'dt_s\'] for r in rows]);H=np.asarray(mapping)\n    world=project(points,H);after=project(next_points,H)\n    origin=np.median(world,axis=0);scale=max(np.ptp(world,axis=0).max(),1)\n    normalizer=np.array([[1/scale,0,-origin[0]/scale],[0,1/scale,-origin[1]/scale],[0,0,1]])\n    w=project(world,normalizer);a=project(after,normalizer)\n    velocity=(a-w)/dt[:,None];median=np.median(velocity,axis=0)\n    speed=max(np.linalg.norm(median),1e-5)\n    def warp(p):return np.array([[1,0,0],[0,1,0],[p[0],p[1],1]])\n    def residual(p):\n        G=warp(p);v=(project(a,G)-project(w,G))/dt[:,None]\n        den=np.c_[np.r_[w,a],np.ones(len(w)*2)]@G[2]\n        return np.r_[((v-p[2:4])/speed).ravel(),p[:2]*.04,np.maximum(.25-den,0)*20]\n    fit=least_squares(residual,np.r_[0,0,median],bounds=([-1.5,-1.5,-20,-20],[1.5,1.5,20,20]),\n                      loss=\'soft_l1\',f_scale=.08,max_nfev=300)\n    G=warp(fit.x);den=np.c_[np.r_[w,a],np.ones(len(w)*2)]@G[2]\n    if np.min(den)<.25:return {\'available\':False,\'reason\':\'refined_mapping_pole\'}\n    refined=np.linalg.inv(normalizer)@G@normalizer@H\n    result=estimate_flow(features,refined)\n    if not result.get(\'available\'):return result\n    result.update(method=\'learned_flow_constant_motion_refinement\',\n        constant_speed_assumed=True,refinement_parameters=fit.x.tolist(),\n        metric_scale_constraint=\'Preserve learned-map Jacobian at support centre\',\n        original_mapping=H.tolist(),velocity_relative_rms=float(np.sqrt(np.mean(residual(fit.x)[:2*len(w)]**2))))\n    return result\n', 'longitudinal_calibration.py': '"""One-dimensional road-distance calibration using reference-pass optical flow.\n\nFor straight road motion with a horizontal horizon, longitudinal distance is\nproportional to reciprocal height above that horizon. A known reference pass\nprovides metric scale; a query video\'s own speed is never an input.\n"""\nimport numpy as np\nfrom scipy.optimize import least_squares\nfrom calibrate_speed import project\nfrom refine_flow import flow_vanishing_point\n\n\ndef fit_longitudinal_reference(flow,reference_kmh):\n    rows=flow.get(\'measurements\',[])\n    if len(rows)<12:return {\'available\':False,\'reason\':\'insufficient_reference_motion\'}\n    vp=flow_vanishing_point(flow)\n    if not vp[\'available\']:return vp\n    t=np.array([r[\'time_s\'] for r in rows]);low,high=np.quantile(t,[.1,.9])\n    rows=[r for r in rows if low<=r[\'time_s\']<=high]\n    p=np.array([r[\'anchor\'] for r in rows]);q=np.array([r[\'transported_anchor\'] for r in rows])\n    dt=np.array([r[\'dt_s\'] for r in rows]);horizon=vp[\'point\'][1]\n    def factors(h):return np.abs(1/(q[:,1]-h)-1/(p[:,1]-h))/dt\n    target=reference_kmh/3.6;initial_scale=target/max(np.median(factors(horizon)),1e-9)\n    max_h=min(np.min(p[:,1]),np.min(q[:,1]))-20\n    lower=min(horizon-80,max_h-150);upper=min(horizon+80,max_h)\n    def residual(parameters):\n        h,loga=parameters\n        return np.r_[(np.exp(loga)*factors(h)/target-1), (h-horizon)/40*.04]\n    fit=least_squares(residual,[min(horizon,upper-1e-5),np.log(initial_scale)],\n        bounds=([lower,np.log(1)],[upper,np.log(1e8)]),loss=\'soft_l1\',f_scale=.05,max_nfev=500)\n    h=float(fit.x[0]);a=float(np.exp(fit.x[1]))\n    # Same median estimator is used for calibration and subsequent queries.\n    a*=target/np.median(a*factors(h))\n    return {\'available\':True,\'horizon_y\':h,\'distance_coefficient_meter_pixels\':a,\n        \'flow_vanishing_point\':vp,\'reference_kmh\':reference_kmh,\n        \'reference_motion_relative_mad\':float(np.median(abs(a*factors(h)-target))/target),\n        \'support_y_pixels\':[float(min(p[:,1].min(),q[:,1].min())),float(max(p[:,1].max(),q[:,1].max()))],\n        \'assumption\':\'Locally straight longitudinal motion; reference-image horizon approximately horizontal\',\n        \'method\':\'reciprocal_road_coordinate_reference_calibration\'}\n\n\ndef predict_longitudinal(flow,calibration,query_to_reference):\n    if not calibration.get(\'available\'):return {\'available\':False,\'reason\':\'reference_calibration_unavailable\'}\n    rows=flow.get(\'measurements\',[])\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_query_flow\'}\n    p=project([r[\'anchor\'] for r in rows],query_to_reference)\n    q=project([r[\'transported_anchor\'] for r in rows],query_to_reference)\n    h=calibration[\'horizon_y\'];dt=np.array([r[\'dt_s\'] for r in rows])\n    t=np.array([r[\'time_s\'] for r in rows]);lo,hi=np.quantile(t,[.1,.9])\n    low,high=calibration[\'support_y_pixels\'];margin=(high-low)*.1\n    keep=(t>=lo)&(t<=hi)&(p[:,1]>h+20)&(q[:,1]>h+20)&(p[:,1]>=low-margin)&(p[:,1]<=high+margin)\n    if keep.sum()<8:return {\'available\':False,\'reason\':\'query_outside_reference_road_support\'}\n    speed=np.abs(1/(q[:,1]-h)-1/(p[:,1]-h))/dt*calibration[\'distance_coefficient_meter_pixels\']*3.6\n    median=np.median(speed[keep]);mad=np.median(abs(speed[keep]-median))\n    keep&=abs(speed-median)<=max(4*1.4826*mad,2)\n    if keep.sum()<8:return {\'available\':False,\'reason\':\'insufficient_consistent_query_flow\'}\n    return {\'available\':True,\'estimated_kmh\':float(np.median(speed[keep])),\n        \'observations\':int(keep.sum()),\'track_id\':flow[\'track_id\'],\n        \'time_interval_s\':[float(t[keep].min()),float(t[keep].max())],\n        \'method\':\'registered_reciprocal_reference_flow\',\'reference_label_used\':False,\n        \'query_to_reference\':np.asarray(query_to_reference).tolist(),\n        \'reference_road_calibration\':calibration,\'per_frame_kmh\':speed.tolist(),\n        \'per_frame_times_s\':t.tolist(),\'per_frame_keep\':keep.tolist()}\n', 'scene_registration.py': '"""Cache speed-blind scene and road correspondences for a reference library."""\nimport cv2\nimport numpy as np\n\n\ndef register_scenes(images):\n    cv2.setRNGSeed(17)\n    names=sorted(images);sift=cv2.SIFT_create(nfeatures=3000,contrastThreshold=.015)\n    features={}\n    for name in names:\n        image=cv2.resize(images[name],(960,540));gray=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)\n        kp,des=sift.detectAndCompute(gray,None)\n        features[name]=(np.array([k.pt for k in kp],np.float32),des)\n    pairs=[]\n    for i,reference in enumerate(names):\n        if features[reference][1] is None:continue\n        matcher=cv2.FlannBasedMatcher(dict(algorithm=1,trees=5),dict(checks=64))\n        matcher.add([features[reference][1]]);matcher.train()\n        for current in names[i+1:]:\n            if features[current][1] is None:continue\n            matches=matcher.knnMatch(features[current][1],k=2)\n            good=[a for pair in matches if len(pair)==2 for a,b in [pair] if a.distance<.7*b.distance]\n            if len(good)<12:continue\n            source=np.float32([features[current][0][m.queryIdx] for m in good])\n            dest=np.float32([features[reference][0][m.trainIdx] for m in good])\n            H,mask=cv2.findHomography(source,dest,cv2.USAC_MAGSAC,1.5)\n            if H is None:continue\n            keep=mask[:,0].astype(bool);count=int(keep.sum())\n            coverage=min(abs(cv2.contourArea(cv2.convexHull(p[keep])))/(960*540) for p in [source,dest]) if count>=3 else 0\n            road=keep&(source[:,1]>320)&(dest[:,1]>320)\n            pairs.append({\'reference\':reference,\'current\':current,\'matches\':len(good),\'inliers\':count,\n                \'ratio\':count/len(good),\'coverage\':coverage,\'road_inliers\':int(road.sum()),\'H_at_960\':H.tolist()})\n    return pairs\n\n\ndef reference_options(query,flow,calibrations,pairs):\n    from longitudinal_calibration import predict_longitudinal\n    options=[]\n    for pair in pairs:\n        if query not in [pair[\'reference\'],pair[\'current\']]:continue\n        if pair[\'inliers\']<50 or pair[\'ratio\']<.6 or pair[\'coverage\']<.2 or pair[\'road_inliers\']<15:continue\n        other=pair[\'reference\'] if query==pair[\'current\'] else pair[\'current\']\n        if other==query or other not in calibrations:continue\n        S=np.diag([.5,.5,1]);T=np.linalg.inv(S)@np.array(pair[\'H_at_960\'])@S\n        if query==pair[\'reference\']:T=np.linalg.inv(T)\n        result=predict_longitudinal(flow,calibrations[other],T)\n        if result.get(\'available\'):\n            options.append({\'reference\':other,\'score\':pair[\'inliers\']*pair[\'coverage\'],\n                \'prediction\':result,\'registration\':pair})\n    return sorted(options,key=lambda r:r[\'score\'],reverse=True)\n', 'calibration_comparison.py': '"""Compare per-video calibration versions and evaluate a reference-library hybrid.\n\nEvery test fold excludes its own speed from calibration and model selection.\nAll clips were previously inspected; these are development cross-validation\nresults, not a newly collected independent accuracy benchmark.\n"""\nimport argparse,csv,hashlib,json,shutil\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom reference_calibration import refine_metric_scale\nfrom longitudinal_calibration import fit_longitudinal_reference\nfrom scene_registration import register_scenes,reference_options\n\nPILOTS=[\'MercedesAMG550_30\',\'MercedesAMG550_60\',\'MercedesAMG550_100\']\nFALLBACK=\'v2d_flow_vp_delay\'\nFUSIONS=[(mode,weight) for mode in [\'best\',\'median\',\'top3mean\',\'top3weighted\'] for weight in [0,.25,.5,.75,1]]\nDEPLOYMENT_AGGREGATION=\'top3weighted\'\n_SCALE_CACHE={}\n\n\ndef metric_scale(raw,labels,version,exclude=()):\n    names=[n for n in PILOTS if n in labels and n not in exclude\n           and raw[n].get(version,{}).get(\'available\')]\n    if not names:return None,[]\n    values=[raw[n][version][\'estimated_kmh\'] for n in names]\n    key=(version,tuple((name,value,labels[name]) for name,value in zip(names,values)))\n    if key not in _SCALE_CACHE:_SCALE_CACHE[key]=refine_metric_scale(values,[labels[n] for n in names])[0]\n    scale=_SCALE_CACHE[key]\n    return scale,names\n\n\ndef predict_version(query,raw,labels,version):\n    result=raw[query].get(version,{\'available\':False,\'reason\':\'missing_version\'})\n    if not result.get(\'available\'):return result\n    scale,names=metric_scale(raw,labels,version,exclude=[query])\n    if scale is None:return {\'available\':False,\'reason\':\'no_metric_scale_reference\'}\n    copied={**result,\'raw_estimated_kmh\':result[\'estimated_kmh\'],\n        \'estimated_kmh\':result[\'estimated_kmh\']*scale,\'metric_scale_multiplier\':scale,\n        \'metric_scale_reference_videos\':names,\'own_reference_speed_used\':False}\n    return copied\n\n\ndef predict_hybrid(query,raw,labels,options,mode,weight):\n    fallback=predict_version(query,raw,labels,FALLBACK)\n    choices=[o for o in options.get(query,[]) if o[\'reference\'] in labels and o[\'reference\']!=query]\n    if not choices or weight==0:\n        return {**fallback,\'calibration_reused\':False,\'library_weight\':0,\'reference_videos\':[],\n                \'calibration_lookup_reason\':\'no_eligible_matching_reference\' if not choices else \'geometry_version_selected\'}\n    choices=sorted(choices,key=lambda o:o[\'score\'],reverse=True)\n    best=choices[0];values=[o[\'prediction\'][\'estimated_kmh\'] for o in choices]\n    if mode==\'best\':reference_estimate=values[0]\n    elif mode==\'median\':reference_estimate=float(np.median(values))\n    elif mode==\'top3mean\':reference_estimate=float(np.mean(values[:3]))\n    elif mode==\'top3weighted\':\n        reference_estimate=float(np.average(values[:3],weights=[o[\'score\'] for o in choices[:3]]))\n    else:raise ValueError(\'Unknown reference aggregation.\')\n    if not fallback.get(\'available\'):weight=1\n    estimate=reference_estimate if weight==1 else weight*reference_estimate+(1-weight)*fallback[\'estimated_kmh\']\n    result={**fallback,**best[\'prediction\'],\'available\':True,\'estimated_kmh\':float(estimate),\n        \'method\':\'reference_library_and_temporal_geometry\',\'calibration_reused\':True,\n        \'library_weight\':weight,\'library_aggregation\':mode,\'reference_component_kmh\':reference_estimate,\n        \'geometry_component_kmh\':fallback.get(\'estimated_kmh\'),\n        \'reference_videos\':[o[\'reference\'] for o in choices],\n        \'individual_reference_predictions_kmh\':values,\'shown_reference_video\':best[\'reference\'],\n        \'registration_evidence\':best[\'registration\'],\'own_reference_speed_used\':False,\n        \'calibration_lookup_reason\':\'verified_scene_overlap_and_supported_longitudinal_path\'}\n    selected=choices[:3] if mode.startswith(\'top3\') else choices[:1]\n    scores=np.array([o[\'score\'] for o in selected],float) if mode==\'top3weighted\' else np.ones(len(selected))\n    scores/=scores.sum()\n    result[\'distance_calibration_components\']=[{\'weight\':float(w),\n        \'query_to_reference\':o[\'prediction\'][\'query_to_reference\'],\n        \'calibration\':o[\'prediction\'][\'reference_road_calibration\']} for o,w in zip(selected,scores)]\n    return result\n\n\ndef select_fold(query,raw,labels,options):\n    # Remove the test reference before calibration or configuration selection.\n    training={name:speed for name,speed in labels.items() if name!=query}\n    scores=[]\n    for mode,weight in FUSIONS:\n        if mode!=DEPLOYMENT_AGGREGATION:continue\n        errors=[]\n        for name,reference in training.items():\n            prediction=predict_hybrid(name,raw,training,options,mode,weight)\n            errors.append(abs(prediction[\'estimated_kmh\']-reference) if prediction.get(\'available\') else 50)\n        scores.append((float(np.mean(errors)),weight,mode))\n    score,weight,mode=min(scores)\n    result=predict_hybrid(query,raw,training,options,mode,weight)\n    result[\'selection\']={\'training_videos\':sorted(training),\'inner_mean_absolute_error_kmh\':score,\n        \'library_weight\':weight,\'library_aggregation\':mode,\'test_reference_excluded\':True}\n    return result\n\n\ndef metrics(rows):\n    errors=np.array([r[\'error_kmh\'] for r in rows if r.get(\'error_kmh\') is not None])\n    return {\'clips\':len(rows),\'estimated\':len(errors),\n        \'mae_kmh\':float(np.mean(abs(errors))) if len(errors) else None,\n        \'rmse_kmh\':float(np.sqrt(np.mean(errors**2))) if len(errors) else None,\n        \'bias_kmh\':float(np.mean(errors)) if len(errors) else None,\n        \'maximum_absolute_error_kmh\':float(np.max(abs(errors))) if len(errors) else None,\n        \'within_2_kmh\':int((abs(errors)<=2).sum()),\'errors_over_10_kmh\':int((abs(errors)>10).sum())}\n\n\ndef run(args):\n    inference=Path(args.inference_dir);first=Path(args.first_round);second=Path(args.second_round);out=Path(args.output)\n    out.mkdir(parents=True,exist_ok=True);raw={};flows={};images={}\n    for folder in sorted(first.iterdir()):\n        if not (folder/\'predictions.json\').exists():continue\n        name=folder.name;raw[name]=json.loads((folder/\'predictions.json\').read_text())\n        raw[name].update(json.loads((second/name/\'predictions.json\').read_text()))\n        flows[name]=json.loads((second/name/\'adaptive_flow_features.json\').read_text())\n        images[name]=cv2.imread(str(inference/name/\'reference.jpg\'))\n    assert len(raw)==30\n    # Reference labels enter the experiment only after speed-blind inference.\n    labels={name:float(name.rsplit(\'_\',1)[1]) for name in raw}\n    calibrations={name:fit_longitudinal_reference(flow,labels[name]) for name,flow in flows.items()}\n    pairs_file=out/\'scene_registrations.json\'\n    pairs=json.loads(pairs_file.read_text()) if pairs_file.exists() else register_scenes(images)\n    pairs_file.write_text(json.dumps(pairs,indent=2),encoding=\'utf-8\')\n    options={name:reference_options(name,flows[name],calibrations,pairs) for name in raw}\n    (out/\'reference_calibrations.json\').write_text(json.dumps(calibrations,indent=2),encoding=\'utf-8\')\n    (out/\'reference_options.json\').write_text(json.dumps(options,indent=2),encoding=\'utf-8\')\n    (out/\'raw_predictions.json\').write_text(json.dumps(raw,indent=2),encoding=\'utf-8\')\n    historical=json.loads(Path(args.historical_results).read_text()) if args.historical_results and Path(args.historical_results).exists() else []\n    historical={Path(r[\'video\']).stem:r for r in historical}\n    rows=[];predictions={};version_rows=[]\n    versions=sorted(set().union(*(p.keys() for p in raw.values())))\n    for name in sorted(raw,key=lambda n:labels[n]):\n        result=select_fold(name,raw,labels,options);predictions[name]=result\n        value=result.get(\'estimated_kmh\');error=None if value is None else value-labels[name]\n        rows.append({\'video\':name+\'.MP4\',\'reference_kmh\':labels[name],\'estimated_kmh\':value,\n            \'previous_estimated_kmh\':historical.get(name,{}).get(\'estimated_kmh\'),\n            \'previous_error_kmh\':historical.get(name,{}).get(\'signed_error_kmh\'),\n            \'error_kmh\':error,\'absolute_error_kmh\':None if error is None else abs(error),\n            \'within_2_kmh\':error is not None and abs(error)<=2,\'method\':result.get(\'method\'),\n            \'calibration_reused\':result.get(\'calibration_reused\',False),\n            \'library_weight\':result.get(\'library_weight\',0),\n            \'reference_videos\':\';\'.join(result.get(\'reference_videos\',[])),\n            \'own_reference_excluded\':True})\n        for version in versions:\n            p=predict_version(name,raw,labels,version);estimate=p.get(\'estimated_kmh\') if p.get(\'available\') else None\n            version_rows.append({\'video\':name+\'.MP4\',\'version\':version,\'reference_kmh\':labels[name],\n                \'estimated_kmh\':estimate,\'error_kmh\':None if estimate is None else estimate-labels[name],\n                \'available\':p.get(\'available\',False)})\n    summary={version:metrics([r for r in version_rows if r[\'version\']==version]) for version in versions}\n    raw_summary={version:metrics([{\'error_kmh\':raw[name][version][\'estimated_kmh\']-labels[name]\n        if raw[name].get(version,{}).get(\'available\') else None} for name in raw]) for version in versions}\n    summary[\'selected_nested_hybrid\']=metrics(rows)\n    summary[\'selected_previous_27\']=metrics([r for r in rows if Path(r[\'video\']).stem not in PILOTS])\n    fusion_summary={}\n    for mode,weight in FUSIONS:\n        values=[]\n        for name in raw:\n            p=predict_hybrid(name,raw,labels,options,mode,weight)\n            values.append({\'error_kmh\':p[\'estimated_kmh\']-labels[name] if p.get(\'available\') else None})\n        fusion_summary[f\'{mode}_weight_{weight}\']=metrics(values)\n    best=min(fusion_summary,key=lambda key:fusion_summary[key][\'mae_kmh\'])\n    source_names=[\'calibration_candidates.py\',\'learned_calibration.py\',\'flow_calibration.py\',\'refine_flow.py\',\n        \'lens_calibration.py\',\'longitudinal_calibration.py\',\'scene_registration.py\',\'calibration_comparison.py\',\n        \'refinement_inference.py\',\'render_comparison.py\']\n    config={\'method_family\':\'Reference library with optical-flow/vanishing-point temporal fallback\',\n        \'selected_development_configuration\':best,\'fallback_version\':FALLBACK,\n        \'nested_test_aggregation\':DEPLOYMENT_AGGREGATION,\n        \'fallback_metric_scale\':metric_scale(raw,labels,FALLBACK)[0],\'metric_scale_reference_videos\':PILOTS,\n        \'scope\':\'This car and reference library; camera correspondence required for reuse\',\n        \'own_reference_excluded_in_each_test\':True,\'fresh_independent_validation\':False,\n        \'source_sha256\':{name:hashlib.sha256((Path(__file__).parent/name).read_bytes()).hexdigest() for name in source_names}}\n    for name,value in [(\'results.json\',rows),(\'predictions.json\',predictions),(\'metrics.json\',summary),\n        (\'fusion_versions.json\',fusion_summary),(\'selected_configuration.json\',config),(\'raw_metrics.json\',raw_summary)]:\n        (out/name).write_text(json.dumps(value,indent=2),encoding=\'utf-8\')\n    for name,value in [(\'results.csv\',rows),(\'version_results.csv\',version_rows)]:\n        with (out/name).open(\'w\',newline=\'\',encoding=\'utf-8\') as stream:\n            writer=csv.DictWriter(stream,fieldnames=list(value[0]));writer.writeheader();writer.writerows(value)\n    if args.archive_inputs:\n        shutil.copytree(first,out/\'inputs/round1\',dirs_exist_ok=True)\n        shutil.copytree(second,out/\'inputs/round2\',dirs_exist_ok=True)\n    make_report(out,rows,summary,fusion_summary,config,raw_summary)\n    if not args.no_render:\n        from render_comparison import render_pair\n        for row in rows:\n            name=Path(row[\'video\']).stem\n            render_pair(Path(args.video_dir)/row[\'video\'],out/\'videos\'/name,inference/name,\n                predictions[name],row[\'reference_kmh\'],flows[name])\n    print(\'Selected nested per-video results:\',summary[\'selected_nested_hybrid\'],flush=True)\n    print(\'Fixed deployment candidate:\',best,fusion_summary[best],flush=True)\n\n\ndef make_report(out,rows,summary,fusions,config,raw_summary):\n    m=summary[\'selected_nested_hybrid\'];comparable=summary[\'selected_previous_27\']\n    lines=[\'# Calibration research and per-video tests\',\'\',\n        \'Run date: 2026-10-16 (Asia/Karachi). All 30 Mercedes clips are tested separately. Text annotations are ignored.\',\n        \'\',f"**Individual test result: MAE {m[\'mae_kmh\']:.2f} km/h; {m[\'within_2_kmh\']}/30 within ±2 km/h; worst error {m[\'maximum_absolute_error_kmh\']:.2f} km/h.**",\'\',\n        \'**These are development reruns of previously inspected videos. They are not a fresh independent validation set. Each test reference is excluded from calibration and automated configuration selection.**\',\'\',\n        \'## Research and implementations\',\'\',\n        \'| Research | Used in this experiment | Scope |\',\'|---|---|---|\',\n        \'| [Revaud and Humenberger, ICCV 2021](https://github.com/naver/cctv) | Released learned mask-moment calibration network, direct and ensemble versions | Pinned upstream code/model; CC BY-NC-SA 3.0 retained; weights not redistributed |\',\n        \'| [Kocur and Ftáčnik, 2021](https://arxiv.org/abs/2103.11438) | Vehicle motion vanishing points constrain new geometric fits | Own optical-flow/line estimator; their trained DeepVP network was not run |\',\n        \'| [Su et al., 2026](https://arxiv.org/abs/2608.16785) | Compare optical flow with keypoint motion and update metric geometry | Inspired hybrid contact-anchor method; not a reproduction of their 36-keypoint network |\',\n        \'| [Fitzgibbon, 2001](https://www.robots.ox.ac.uk/~vgg/publications/2001/Fitzgibbon01b/fitzgibbon01b.pdf) | Division-model radial correction estimated from visual motion | Own nonlinear motion fit; not the paper’s two-view eigenvalue algorithm |\',\n        \'| [Vuong et al., WACV 2024](https://github.com/kvuong2711/OpenTrafficCam3D) | Considered scene-reconstruction calibration | Not run: required street-level/GPS imagery is absent |\',\'\',\n        \'Additional independently implemented candidates fit rigid cuboid motion, generic vehicle silhouettes, temporal front/rear delays, and reference-pass reciprocal road coordinates. No neural-network weights were fine-tuned. Vehicle dimensions remain priors, not a confirmed Mercedes CAD model.\',\'\',\n        \'## Pattern analysis\',\'\',\n        \'The historical 27-clip errors were not concentrated only at high speed: MAE was 7.49 km/h below 50, 11.64 at 50–79, and 8.17 at 80+. Correlation between reference speed and absolute error was approximately 0.086. These descriptive results do not establish causation. Unmatched camera views and inconsistent predicted hidden corners were stronger concerns.\',\'\',\n        \'The first GPU round compared ten versions on all 30 clips. Small image-fitting residuals did not guarantee accurate speed: the silhouette fits frequently selected incorrect focal lengths and scales. The learned network fitted the three pilots well after scale correction but transferred poorly to some other views. A two-dimensional projective time-delay method was more consistent. The second round added shared motion vanishing points, adaptive frame gaps, constant-motion rectification, a constrained 3D fit, and estimated radial distortion.\',\'\',\n        \'## Individual test protocol\',\'\',\n        \'1. Produce speed-blind geometry, mask moments and optical-flow measurements for every video.\',\n        \'2. For each test, remove its reference speed before fitting metric scale or selecting a configuration. The fixed 30/60/100 calibration pilots are used only when they remain in that fold.\',\n        \'3. Each eligible other video can supply a known-speed reference calibration. Scene matching and supported image locations decide whether it can be reused. A test is never its own reference.\',\n        \'4. Combine up to three strongest matching references, weighted by visual overlap. This aggregation reduces single-reference switching. Select its reference/geometry fusion weight using inner tests on the remaining 29 videos, excluding each inner test from its own references as well. Then predict the outer test.\',\n        \'5. Read the outer reference only to calculate its error. Retain all failures and unavailable results. This is internal development cross-validation on one car and location.\',\'\',\n        \'Reference-library results benefit from additional labelled reference passes. They must not be attributed solely to a new automatic calibration model. Raw geometry/model predictions are reported separately from pilot scale correction and from reference-library assistance.\',\'\',\n        \'## Every geometric/model version\',\'\',\n        \'Each row uses reference scale fitted from the designated pilots with that test excluded. Missing estimates count against coverage and the tolerance total.\',\'\',\n        \'| Version | Estimated / 30 | Raw MAE | Scale-corrected MAE | Worst corrected error | Within ±2 | Errors >10 |\',\n        \'|---|---:|---:|---:|---:|---:|---:|\']\n    for version,result in summary.items():\n        if version.startswith(\'selected_\'):continue\n        def fmt(x):return \'unavailable\' if x is None else f\'{x:.2f}\'\n        lines.append(f"| {version} | {result[\'estimated\']}/30 | {fmt(raw_summary[version][\'mae_kmh\'])} | {fmt(result[\'mae_kmh\'])} | {fmt(result[\'maximum_absolute_error_kmh\'])} | {result[\'within_2_kmh\']} | {result[\'errors_over_10_kmh\']} |")\n    lines+=[\'\',\'## Hybrid versions\',\'\',\n        \'Weights are the contribution of the reference-library estimate when a compatible reference exists; otherwise the vanishing-point temporal estimate is used. These fixed-version rows are development comparisons. Final individual tests use the weighted top-three reference family and select only its fusion weight within each training fold.\',\'\',\n        \'| Version | MAE km/h | Worst error | Within ±2 |\',\'|---|---:|---:|---:|\']\n    for version,result in fusions.items():\n        lines.append(f"| {version} | {result[\'mae_kmh\']:.2f} | {result[\'maximum_absolute_error_kmh\']:.2f} | {result[\'within_2_kmh\']}/30 |")\n    lines+=[\'\',f"The frozen development configuration is **{config[\'selected_development_configuration\']}**. Its settings are in [selected_configuration.json](selected_configuration.json).",\'\',\n        \'## Before and after\',\'\',\n        f"On the same previous 27 test clips, historical MAE was 9.41 km/h and this development cross-validation gives {comparable[\'mae_kmh\']:.2f} km/h. The new hybrid can use other labelled reference passes, while the historical run used only three pilots. The datasets and training information therefore differ; this is not a new independent head-to-head validation.",\'\',\n        \'![Historical and new individual errors](errors.jpg)\',\'\',\n        \'## Each video and its two outputs\',\'\',\n        \'| Video / calibration | Cruise reference | Previous estimate | New estimate | New signed error | Within ±2 | Calibration reused |\',\n        \'|---|---:|---:|---:|---:|---|---|\']\n    for row in rows:\n        stem=Path(row[\'video\']).stem\n        previous=\'unavailable\' if row[\'previous_estimated_kmh\'] is None else f"{row[\'previous_estimated_kmh\']:.1f}"\n        lines.append(f"| [{row[\'video\']}](videos/{stem}/speed.mp4) · [calibration](videos/{stem}/calibration.mp4) | {row[\'reference_kmh\']:.1f} | {previous} | {row[\'estimated_kmh\']:.1f} | {row[\'error_kmh\']:+.2f} | {\'Yes\' if row[\'within_2_kmh\'] else \'No\'} | {\'Yes\' if row[\'calibration_reused\'] else \'No\'} |")\n    lines+=[\'\',\'## Files and reproduction\',\'\',\n        \'- [Per-video CSV](results.csv), [all version results](version_results.csv), [metrics](metrics.json), [raw metrics before reference scaling](raw_metrics.json), [fold predictions and reference lists](predictions.json).\',\n        \'- [Scene-registration evidence](scene_registrations.json), [reference calibrations](reference_calibrations.json), [reference options](reference_options.json), [raw version predictions](raw_predictions.json).\',\n        \'- `inputs/round1` and `inputs/round2` preserve the model/motion outputs. Original videos and all downloaded weights are excluded.\',\n        \'- Calibration videos review predicted nodes and the fitted longitudinal mapping. They do not claim a measured reconstruction of the entire road.\',\n        \'- Reproduce from the feature folder using `python calibration_comparison.py --inference-dir output/mercedes_validation/inference --first-round output/calibration_comparison/inputs/round1 --second-round output/calibration_comparison/inputs/round2 --output outputs/comparison_reproduction --no-render`. Add `--video-dir <original-folder>` and omit `--no-render` to render videos.\',\'\',\n        \'## Accuracy limit\',\'\',\n        \'The requested ±1–2 km/h accuracy is achieved only if every individual error supports it. The report preserves the remaining large errors. Cruise-control settings are reference values, not independently measured radar speeds. Reference-library transfer assumes a compatible local road projection; full-scene matching can still contain ground parallax or lens effects. Unknown views rely on approximate vehicle geometry. The Mercedes calibration must not be copied to the original CCTV camera.\']\n    if m[\'maximum_absolute_error_kmh\']>2:lines+=[\'\',\'**The requested accuracy target is still not met across all videos.**\']\n    (out/\'README.md\').write_text(\'\\n\'.join(lines)+\'\\n\',encoding=\'utf-8\')\n    fig,axes=plt.subplots(1,2,figsize=(13,5))\n    refs=[r[\'reference_kmh\'] for r in rows];values=[r[\'estimated_kmh\'] for r in rows];errors=[r[\'error_kmh\'] for r in rows]\n    if all(r[\'previous_estimated_kmh\'] is not None for r in rows):\n        axes[0].scatter(refs,[r[\'previous_estimated_kmh\'] for r in rows],c=\'grey\',marker=\'x\',label=\'Historical run\')\n        axes[1].scatter(refs,[r[\'previous_error_kmh\'] for r in rows],c=\'grey\',marker=\'x\',label=\'Historical run\')\n    axes[0].scatter(refs,values,c=[\'steelblue\' if r[\'calibration_reused\'] else \'darkorange\' for r in rows],label=\'New individual tests\')\n    axes[0].plot([25,110],[25,110],\'k--\');axes[0].set(xlabel=\'Cruise reference km/h\',ylabel=\'Estimated km/h\',title=\'Each video excluded from its own references\')\n    axes[1].axhspan(-2,2,color=\'green\',alpha=.15);axes[1].scatter(refs,errors,label=\'New individual tests\')\n    axes[1].axhline(0,color=\'black\',lw=.5);axes[1].set(xlabel=\'Cruise reference km/h\',ylabel=\'Signed error km/h\',title=\'All individual errors retained\')\n    for axis in axes:axis.grid(alpha=.2);axis.legend()\n    fig.tight_layout();fig.savefig(out/\'errors.jpg\',dpi=150);plt.close(fig)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    for name in [\'inference-dir\',\'first-round\',\'second-round\',\'output\']:parser.add_argument(\'--\'+name,required=True)\n    parser.add_argument(\'--video-dir\');parser.add_argument(\'--no-render\',action=\'store_true\')\n    parser.add_argument(\'--historical-results\',default=str(Path(__file__).parent/\'output/mercedes_validation/results.json\'))\n    parser.add_argument(\'--archive-inputs\',action=\'store_true\');run(parser.parse_args())\n', 'shared_camera_calibration.py': '"""Reference-assisted camera maps fitted jointly across known-speed passes.\n\nThe input speed dictionary contains calibration passes only. Query speed is never\nan argument. A camera map is reused after lower-image scene registration; each\nprediction stays within the calibrated image support. These models assume a\nstraight, approximately horizontal road and constant speed during each pass.\n"""\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\nfrom calibrate_speed import project\n\n\ndef foreground_registration(reference, current):\n    """Align lower-image ground texture, avoiding a homography dominated by sky."""\n    cv2.setRNGSeed(17)\n    a = cv2.resize(reference, (960, 540))\n    b = cv2.resize(current, (960, 540))\n    mask = np.zeros((540, 960), np.uint8)\n    mask[300:530, 15:945] = 255\n    sift = cv2.SIFT_create(nfeatures=6000, contrastThreshold=.01)\n    ka, da = sift.detectAndCompute(cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), mask)\n    kb, db = sift.detectAndCompute(cv2.cvtColor(b, cv2.COLOR_BGR2GRAY), mask)\n    if da is None or db is None:\n        return {\'accepted\': False, \'reason\': \'insufficient_foreground_texture\'}\n    pairs = cv2.BFMatcher().knnMatch(db, da, k=2)\n    good = [x for pair in pairs if len(pair) == 2 for x, y in [pair] if x.distance < .7*y.distance]\n    if len(good) < 15:\n        return {\'accepted\': False, \'reason\': \'insufficient_foreground_matches\', \'matches\': len(good)}\n    p = np.float32([kb[m.queryIdx].pt for m in good])\n    q = np.float32([ka[m.trainIdx].pt for m in good])\n    H, inliers = cv2.findHomography(p, q, cv2.USAC_MAGSAC, 1.5)\n    if H is None:\n        return {\'accepted\': False, \'reason\': \'foreground_homography_failed\'}\n    keep = inliers[:, 0].astype(bool)\n    count = int(keep.sum())\n    coverage = min(abs(cv2.contourArea(cv2.convexHull(x[keep])))/(960*540) for x in [p,q]) if count >= 3 else 0\n    rms = float(np.sqrt(np.mean(np.sum((project(p[keep], H)-q[keep])**2, axis=1))))\n    accepted = count >= 20 and count/len(good) >= .6 and coverage >= .025 and rms <= 1.5\n    S = np.diag([.5, .5, 1])\n    return {\'accepted\': bool(accepted), \'matches\': len(good), \'inliers\': count,\n            \'coverage\': coverage, \'rms_pixels_at_960\': rms,\n            \'query_to_template\': (np.linalg.inv(S)@H@S).tolist(),\n            \'scope\': \'Lower-image planar texture; this heuristic is not semantic road segmentation.\'}\n\n\ndef aligned_measurements(flow, transform):\n    rows = flow.get(\'measurements\', [])\n    if len(rows) < 8:\n        return None\n    p = project([r[\'anchor\'] for r in rows], np.asarray(transform))\n    q = project([r[\'transported_anchor\'] for r in rows], np.asarray(transform))\n    t = np.array([r[\'time_s\'] for r in rows])\n    dt = np.array([r[\'dt_s\'] for r in rows])\n    keep = np.isfinite(p).all(axis=1) & np.isfinite(q).all(axis=1) & (dt > 0)\n    keep &= np.array([r.get(\'inliers\', 0) >= 6 for r in rows])\n    keep &= np.array([r.get(\'flow_rms_pixels\', 0) <= 2 for r in rows])\n    if keep.sum() < 8:\n        return None\n    low, high = np.quantile(t[keep], [.1, .9])\n    keep &= (t >= low) & (t <= high)\n    if keep.sum() < 8:\n        return None\n    return {\'p\': p[keep], \'q\': q[keep], \'t\': t[keep], \'dt\': dt[keep],\n            \'track_id\': flow.get(\'track_id\')}\n\n\ndef corrected_points(points, radial_lambda=0):\n    points = np.asarray(points)\n    offset = (points-[960, 540])/1920\n    denominator = 1+radial_lambda*np.sum(offset**2, axis=1)\n    return offset/denominator[:, None]*1920+[960, 540]\n\n\ndef camera_coordinate(points, model):\n    p = corrected_points(points, model.get(\'radial_lambda\', 0))\n    y = p[:, 1]\n    if model[\'family\'] == \'scale_curve\':\n        return np.interp(y, model[\'coordinate_y_pixels\'], model[\'coordinate_meters\'])\n    return model[\'distance_coefficient_meter_pixels\']/(y-model[\'horizon_y\'])\n\n\ndef _reference_data(flows, speeds, transforms):\n    data = []\n    for name in sorted(speeds):\n        if name not in flows or name not in transforms:\n            continue\n        r = aligned_measurements(flows[name], transforms[name])\n        if r is not None:\n            data.append({**r, \'video\': name, \'reference_mps\': speeds[name]/3.6})\n    return data\n\n\ndef fit_shared_camera(flows, speeds, transforms, family=\'trajectory\', degree=2):\n    """Fit a camera using reference speeds only, balancing contributions per pass."""\n    data = _reference_data(flows, speeds, transforms)\n    if len(data) < 2:\n        return {\'available\': False, \'reason\': \'fewer_than_two_compatible_calibration_passes\'}\n    all_y = np.concatenate([r[\'p\'][:, 1] for r in data])\n    low_y, high_y = float(all_y.min()), float(all_y.max())\n    if family == \'scale_curve\':\n        center = float(np.median(all_y)); scale = max(float(np.ptp(all_y))/2, 50)\n        ys=[]; targets=[]; weights=[]\n        for row in data:\n            displacement=np.abs(row[\'q\'][:,1]-row[\'p\'][:,1])/row[\'dt\']\n            keep=displacement>.5\n            ys.extend(row[\'p\'][keep,1]); targets.extend(np.log(row[\'reference_mps\']/displacement[keep]))\n            weights.extend([1/np.sqrt(max(keep.sum(),1))]*int(keep.sum()))\n        ys=np.asarray(ys);targets=np.asarray(targets);weights=np.asarray(weights)\n        if len(ys)<20:return {\'available\':False,\'reason\':\'insufficient_curve_samples\'}\n        X=np.vander((ys-center)/scale,degree+1,increasing=True)\n        initial=np.linalg.lstsq(X*weights[:,None],targets*weights,rcond=None)[0]\n        def residual(c):return np.r_[(X@c-targets)*weights, c[2:]*.025]\n        fit=least_squares(residual,initial,loss=\'soft_l1\',f_scale=.03,max_nfev=500)\n        grid=np.linspace(low_y,high_y,512)\n        meters_per_pixel=np.exp(np.vander((grid-center)/scale,degree+1,increasing=True)@fit.x)\n        coordinate=np.r_[0,np.cumsum((meters_per_pixel[:-1]+meters_per_pixel[1:])*.5*np.diff(grid))]\n        return {\'available\':True,\'family\':family,\'degree\':degree,\'curve_coefficients\':fit.x.tolist(),\n                \'curve_center_y\':center,\'curve_scale_y\':scale,\'coordinate_y_pixels\':grid.tolist(),\n                \'coordinate_meters\':coordinate.tolist(),\'meters_per_y_pixel\':meters_per_pixel.tolist(),\n                \'support_y_pixels\':[low_y,high_y],\'reference_videos\':[r[\'video\'] for r in data],\n                \'reference_label_used\':True,\'query_reference_label_used\':False}\n    radial = family == \'trajectory_radial\'\n    upper_h = low_y-20\n    initial_h = min(500, upper_h-30)\n    factors = []\n    for row in data:\n        c=1/(row[\'p\'][:,1]-initial_h)\n        slope=abs(np.polyfit(row[\'t\'],c,1)[0])\n        if slope>1e-8:factors.append(row[\'reference_mps\']/slope)\n    initial_a=float(np.median(factors)) if factors else 10000\n    initial=[initial_h,np.log(np.clip(initial_a,1,1e8))]+([0] if radial else [])\n    lower=[-540,np.log(1)]+([-.75] if radial else [])\n    upper=[upper_h,np.log(1e8)]+([.75] if radial else [])\n    def residual(parameters):\n        h,loga=parameters[:2];a=np.exp(loga);lam=parameters[2] if radial else 0\n        result=[]\n        for row in data:\n            p=corrected_points(row[\'p\'],lam);den=p[:,1]-h\n            if family == \'flow\':\n                q=corrected_points(row[\'q\'],lam)\n                v=a*np.abs(1/np.maximum(q[:,1]-h,10)-1/np.maximum(den,10))/row[\'dt\']\n                r=v/row[\'reference_mps\']-1\n            else:\n                z=a/np.maximum(den,10);t=row[\'t\']-row[\'t\'].mean()\n                sign=np.sign(np.polyfit(t,z,1)[0]) or 1\n                r=(z-z.mean()-sign*row[\'reference_mps\']*t)/row[\'reference_mps\']\n            result.extend(r/np.sqrt(len(r)))\n            result.extend(np.minimum(den-10,0)/np.sqrt(len(den)))\n        if radial:result.append(lam*.002)\n        return np.asarray(result)\n    fit=least_squares(residual,initial,bounds=(lower,upper),loss=\'soft_l1\',\n                      f_scale=.01 if family!=\'flow\' else .04,max_nfev=1000)\n    lam=float(fit.x[2]) if radial else 0\n    bound=radial and abs(lam)>.745\n    coordinates=np.concatenate([corrected_points(r[\'p\'],lam)[:,1] for r in data])\n    if min(coordinates)-fit.x[0]<15:return {\'available\':False,\'reason\':\'mapping_pole_near_calibrated_path\'}\n    return {\'available\':not bound,\'reason\':\'radial_parameter_at_bound\' if bound else \'fitted\',\n            \'family\':family,\'horizon_y\':float(fit.x[0]),\n            \'distance_coefficient_meter_pixels\':float(np.exp(fit.x[1])),\n            \'radial_lambda\':lam,\'support_y_pixels\':[float(coordinates.min()),float(coordinates.max())],\n            \'reference_videos\':[r[\'video\'] for r in data],\n            \'fit_residual_rms\':float(np.sqrt(np.mean(residual(fit.x)**2))),\n            \'reference_label_used\':True,\'query_reference_label_used\':False,\n            \'assumption\':\'One shared longitudinal camera map, approximately horizontal road horizon.\'}\n\n\ndef predict_shared_camera(flow, model, transform):\n    if not model.get(\'available\'):\n        return {\'available\':False,\'reason\':model.get(\'reason\',\'unavailable_camera_map\')}\n    row=aligned_measurements(flow,transform)\n    if row is None:return {\'available\':False,\'reason\':\'insufficient_query_measurements\'}\n    p=row[\'p\'];q=row[\'q\'];t=row[\'t\'];dt=row[\'dt\']\n    y=corrected_points(p,model.get(\'radial_lambda\',0))[:,1]\n    low,high=model[\'support_y_pixels\'];margin=(high-low)*.05\n    keep=(y>=low-margin)&(y<=high+margin)\n    if model[\'family\']==\'scale_curve\':keep=(y>=low)&(y<=high)\n    if keep.sum()<8:return {\'available\':False,\'reason\':\'query_outside_shared_camera_support\'}\n    z=camera_coordinate(p,model)\n    if model[\'family\'] in [\'flow\',\'scale_curve\']:\n        after=camera_coordinate(q,model);values=np.abs(after-z)/dt*3.6\n        median=np.median(values[keep]);mad=np.median(abs(values[keep]-median))\n        keep&=abs(values-median)<=max(4*1.4826*mad,2)\n        if keep.sum()<8:return {\'available\':False,\'reason\':\'inconsistent_query_flow\'}\n        speed=float(np.median(values[keep]));rms=None\n    else:\n        times=t[keep]-t[keep].mean();positions=z[keep]\n        initial=np.polyfit(times,positions,1)\n        fit=least_squares(lambda p:p[0]*times+p[1]-positions,initial,loss=\'soft_l1\',f_scale=.15,max_nfev=300)\n        speed=float(abs(fit.x[0])*3.6)\n        rms=float(np.sqrt(np.mean((fit.x[0]*times+fit.x[1]-positions)**2)))\n    return {\'available\':True,\'estimated_kmh\':speed,\'method\':\'shared_camera_\'+model[\'family\'],\n            \'track_id\':row[\'track_id\'],\'time_interval_s\':[float(t[keep].min()),float(t[keep].max())],\n            \'observations\':int(keep.sum()),\'trajectory_rms_m\':rms,\'calibration_reused\':True,\n            \'reference_videos\':model[\'reference_videos\'],\'shared_camera_model\':model,\n            \'query_to_template\':np.asarray(transform).tolist(),\'query_reference_speed_used\':False,\n            \'own_reference_excluded\':True}\n', 'vehicle_geometry_refinement.py': '"""Speed-blind SUV geometry and temporal-window ablations, without a fixed flow VP."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport cv2\nfrom benchmark_speed import load_clip\nfrom calibration_candidates import select_rows, temporal_delay_2d, rigid_cuboid_bundle, shared_vp_delay\n\n\ndef run(args):\n    cv2.setNumThreads(4)\n    out=Path(args.output);out.mkdir(parents=True,exist_ok=True)\n    for folder in sorted(Path(args.inference_dir).iterdir()):\n        if not (folder/\'observations.json\').exists():continue\n        target=out/folder.name;target.mkdir(exist_ok=True)\n        observations,report=load_clip(folder)\n        predictions={}\n        for label,minimum,maximum in [(\'far\',60,250),(\'middle\',80,350),(\'wide\',80,450)]:\n            predictions[\'delay_\'+label]=temporal_delay_2d(select_rows(observations,report,minimum,maximum))\n        for distortion in [False,True]:\n            label=\'suv_rigid_radial\' if distortion else \'suv_rigid\'\n            predictions[label]=rigid_cuboid_bundle(observations,report,length_m=4.485,width_m=1.855,\n                                                  height_m=1.645,distortion=distortion)\n        predictions[\'corner_vp_weak\']=shared_vp_delay(select_rows(observations,report),vp_weight=.002)\n        (target/\'predictions.json\').write_text(json.dumps(predictions,indent=2)+\'\\n\',encoding=\'utf-8\')\n        print(folder.name,{k:round(v[\'estimated_kmh\'],3) if v.get(\'available\') else None for k,v in predictions.items()},flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--inference-dir\',required=True);parser.add_argument(\'--output\',required=True)\n    run(parser.parse_args())\n', 'kia_validation.py': '"""Cross-vehicle development and a locked, reserved Kia speed evaluation.\n\nDevelopment reads eight Kia references. Reserved speed errors cannot be generated\nuntil the chosen method and its source hashes have been frozen to configuration.\nThe previous Mercedes calibration is retained as a separate transfer baseline.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nfrom calibration_comparison import predict_hybrid, predict_version, metrics, FALLBACK\nfrom longitudinal_calibration import fit_longitudinal_reference\nfrom scene_registration import register_scenes, reference_options\nfrom shared_camera_calibration import foreground_registration, fit_shared_camera, predict_shared_camera\n\nROOT=Path(__file__).resolve().parent\nCANDIDATES=[\'vehicle_length_prior\',\'expanded_reference_library\',\'foreground_reference_library\',\n            \'shared_flow\',\'shared_trajectory\',\'shared_trajectory_radial\',\'shared_curve2\',\'shared_curve3\',\n            \'geometry_native\',\'geometry_suv\',\'geometry_temporal\',\n            \'geometry_ensemble_25\',\'geometry_ensemble_50\',\'geometry_ensemble_75\']\nSOURCE_FILES=[\'kia_validation.py\',\'shared_camera_calibration.py\',\'calibration_comparison.py\',\n              \'longitudinal_calibration.py\',\'scene_registration.py\',\'refine_flow.py\',\'calibration_candidates.py\',\n              \'vehicle_geometry_refinement.py\']\nKIA_LENGTH_PRIOR_M=4.485\n\n\ndef read(path):return json.loads(Path(path).read_text(encoding=\'utf-8\'))\ndef write(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    path.write_text(json.dumps(value,indent=2)+\'\\n\',encoding=\'utf-8\')\n\n\ndef source_hashes():return {name:hashlib.sha256((ROOT/name).read_bytes()).hexdigest() for name in SOURCE_FILES}\n\n\ndef load_measurements(folder):\n    folder=Path(folder);raw={};flows={};images={};inference={}\n    for clip in sorted((folder/\'inference\').iterdir()):\n        if not (clip/\'observations.json\').exists():continue\n        name=clip.name\n        raw[name]={**read(folder/\'round1\'/name/\'predictions.json\'),**read(folder/\'round2\'/name/\'predictions.json\')}\n        extra=folder/\'geometry_refinement\'/name/\'predictions.json\'\n        if extra.exists():raw[name].update(read(extra))\n        flows[name]=read(folder/\'round2\'/name/\'adaptive_flow_features.json\')\n        images[name]=cv2.imread(str(clip/\'reference.jpg\'));assert images[name] is not None\n        inference[name]=str(clip)\n    return raw,flows,images,inference\n\n\ndef load_mercedes():\n    saved=ROOT/\'output/calibration_comparison\'\n    raw=read(saved/\'raw_predictions.json\')\n    flows={name:read(saved/\'inputs/round2\'/name/\'adaptive_flow_features.json\') for name in raw}\n    inference={name:str(ROOT/\'output/mercedes_validation/inference\'/name) for name in raw}\n    images={name:cv2.imread(str(Path(inference[name])/\'reference.jpg\')) for name in raw}\n    return raw,flows,images,inference\n\n\ndef foreground_pairs(images, queries, references, cache_file):\n    cache=read(cache_file) if Path(cache_file).exists() else {}\n    for query in sorted(queries):\n        for reference in sorted(references):\n            if query==reference:continue\n            key=query+\'|\'+reference\n            inverse=reference+\'|\'+query\n            if key in cache:continue\n            if inverse in cache and cache[inverse].get(\'accepted\'):\n                old=cache[inverse]\n                cache[key]={**old,\'query_to_template\':np.linalg.inv(old[\'query_to_template\']).tolist(),\n                            \'inverted_cached_registration\':True}\n            else:\n                # The template is the query image. This transform maps reference pixels to it.\n                cache[key]=foreground_registration(images[query],images[reference])\n        write(cache_file,cache)\n        print(\'Foreground registrations:\',query,flush=True)\n    return cache\n\n\ndef foreground_options(query, flow, calibrations, cache):\n    from longitudinal_calibration import predict_longitudinal\n    values=[]\n    for reference,calibration in calibrations.items():\n        if reference==query:continue\n        evidence=cache.get(query+\'|\'+reference,{})\n        if not evidence.get(\'accepted\'):continue\n        transform=np.linalg.inv(evidence[\'query_to_template\'])\n        prediction=predict_longitudinal(flow,calibration,transform)\n        if prediction.get(\'available\'):\n            values.append({\'reference\':reference,\'prediction\':prediction,\n                           \'score\':evidence[\'inliers\']*evidence[\'coverage\'],\n                           \'registration\':evidence})\n    return sorted(values,key=lambda r:r[\'score\'],reverse=True)\n\n\ndef candidate(query, method, raw, flows, training, options, foreground, cache, baseline):\n    allowed={name:speed for name,speed in training.items() if name!=query}\n    if method.startswith(\'geometry_\'):\n        # Empirical template scale is vehicle-specific, not a camera homography.\n        # It can transfer across viewpoints while each query fits its own camera.\n        same_car={name:speed for name,speed in allowed.items() if name.rsplit(\'_\',1)[0]==query.rsplit(\'_\',1)[0]}\n        families={\'geometry_native\':[(\'v3b_rigid_distortion\',1)],\n                  \'geometry_suv\':[(\'suv_rigid_radial\',1)],\n                  \'geometry_temporal\':[(\'v2b_delay_2d\',1)],\n                  \'geometry_ensemble_25\':[(\'v3b_rigid_distortion\',.25),(\'suv_rigid_radial\',.75)],\n                  \'geometry_ensemble_50\':[(\'v3b_rigid_distortion\',.5),(\'suv_rigid_radial\',.5)],\n                  \'geometry_ensemble_75\':[(\'v3b_rigid_distortion\',.75),(\'suv_rigid_radial\',.25)]}\n        parts=[]\n        for version,weight in families[method]:\n            source=raw[query].get(version,{})\n            references=[name for name in same_car if raw[name].get(version,{}).get(\'available\')]\n            if not source.get(\'available\') or len(references)<2:continue\n            ratios=[same_car[name]/raw[name][version][\'estimated_kmh\'] for name in references]\n            scale=float(np.median(ratios))\n            parts.append({\'version\':version,\'weight\':weight,\'metric_scale\':scale,\n                          \'prediction\':source,\'reference_videos\':references,\n                          \'estimated_kmh\':source[\'estimated_kmh\']*scale})\n        if not parts:return {**baseline,\'adaptation\':method,\'used_frozen_baseline_fallback\':True}\n        denominator=sum(p[\'weight\'] for p in parts)\n        for part in parts:part[\'weight\']/=denominator\n        dominant=max(parts,key=lambda p:p[\'weight\'])[\'prediction\']\n        estimate=float(sum(p[\'weight\']*p[\'estimated_kmh\'] for p in parts))\n        disagreement=float(np.ptp([p[\'estimated_kmh\'] for p in parts]))\n        return {**dominant,\'estimated_kmh\':estimate,\n                \'method\':\'per_video_geometry_with_reference_template_scale\',\'adaptation\':method,\n                \'reference_videos\':sorted(set(n for p in parts for n in p[\'reference_videos\'])),\n                \'geometry_calibration_components\':parts,\'calibration_reused\':False,\n                \'template_disagreement_kmh\':disagreement,\n                \'calibration_review_required\':disagreement>max(10,estimate*.15),\n                \'disagreement_is_accuracy_interval\':False,\n                \'own_reference_excluded\':True,\'query_reference_speed_used\':False,\n                \'scale_scope\':\'Empirical cuboid-template bias correction for this vehicle family; query camera fitted independently.\'}\n    if method==\'vehicle_length_prior\':\n        result={**baseline}\n        if not baseline.get(\'calibration_reused\'):\n            result[\'estimated_kmh\']*=KIA_LENGTH_PRIOR_M/4.9\n        else:\n            result[\'estimated_kmh\']=baseline[\'library_weight\']*baseline[\'reference_component_kmh\']+(1-baseline[\'library_weight\'])*baseline[\'geometry_component_kmh\']*KIA_LENGTH_PRIOR_M/4.9\n            result[\'geometry_component_kmh\']*=KIA_LENGTH_PRIOR_M/4.9\n        return {**result,\'method\':\'frozen_baseline_with_vehicle_length_prior\',\'length_prior_m\':KIA_LENGTH_PRIOR_M}\n    if method in [\'expanded_reference_library\',\'foreground_reference_library\']:\n        choices=options if method==\'expanded_reference_library\' else foreground\n        return {**predict_hybrid(query,raw,allowed,choices,\'top3weighted\',.75),\'adaptation\':method}\n    transforms={name:evidence[\'query_to_template\'] for name in allowed\n                if (evidence:=cache.get(query+\'|\'+name,{})).get(\'accepted\')}\n    family={\'shared_flow\':\'flow\',\'shared_trajectory\':\'trajectory\',\'shared_trajectory_radial\':\'trajectory_radial\',\n            \'shared_curve2\':\'scale_curve\',\'shared_curve3\':\'scale_curve\'}[method]\n    model=fit_shared_camera(flows,allowed,transforms,family,degree=3 if method.endswith(\'3\') else 2)\n    result=predict_shared_camera(flows[query],model,np.eye(3))\n    if not result.get(\'available\'):\n        result={**baseline,\'shared_camera_unavailable_reason\':result.get(\'reason\'),\n                \'shared_camera_attempt\':model,\'used_frozen_baseline_fallback\':True}\n    return {**result,\'adaptation\':method,\'own_reference_excluded\':True}\n\n\ndef row(name, result, speed, method, phase):\n    estimate=result.get(\'estimated_kmh\') if result.get(\'available\') else None\n    error=None if estimate is None else estimate-speed\n    return {\'video\':name+\'.MP4\',\'phase\':phase,\'version\':method,\'reference_kmh\':speed,\'estimated_kmh\':estimate,\n            \'error_kmh\':error,\'absolute_error_kmh\':None if error is None else abs(error),\n            \'within_2_kmh\':error is not None and abs(error)<=2,\n            \'reference_videos\':\';\'.join(result.get(\'reference_videos\',[])),\n            \'calibration_reused\':result.get(\'calibration_reused\',False),\n            \'baseline_fallback\':result.get(\'used_frozen_baseline_fallback\',False)}\n\n\ndef export_rows(path,rows):\n    with Path(path).open(\'w\',encoding=\'utf-8\',newline=\'\') as stream:\n        writer=csv.DictWriter(stream,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)\n\n\ndef run(args):\n    cv2.setNumThreads(4)\n    plan=read(ROOT/\'kia_experiment_plan.json\')\n    out=Path(args.output);out.mkdir(parents=True,exist_ok=True)\n    config_path=Path(args.configuration)\n    config=None\n    if args.phase==\'reserved\':\n        config=read(config_path)\n        assert config[\'frozen_before_reserved_errors\'] is True\n        assert config[\'source_sha256\']==source_hashes(),\'Calibration code changed after configuration freeze.\'\n        assert config[\'calibration_kia_videos\']==plan[\'development_videos\']\n    raw,flows,images,inference=load_mercedes()\n    mercedes_labels={name:float(name.rsplit(\'_\',1)[1]) for name in raw}\n    for folder in [args.development_measurements]+([args.reserved_measurements] if args.phase==\'reserved\' else []):\n        r,f,i,d=load_measurements(folder);raw.update(r);flows.update(f);images.update(i);inference.update(d)\n    queries=plan[\'development_videos\'] if args.phase==\'development\' else plan[\'reserved_test_videos\']\n    assert all(name in raw for name in queries)\n    # Only designated development speeds enter the reference library.\n    development_labels={name:float(name.rsplit(\'_\',1)[1]) for name in plan[\'development_videos\']}\n    training={**mercedes_labels,**development_labels}\n    calibrations={name:fit_longitudinal_reference(flows[name],speed) for name,speed in training.items()}\n    global_file=out/\'global_scene_registrations.json\'\n    pairs=read(global_file) if global_file.exists() else register_scenes(images)\n    write(global_file,pairs)\n    options={name:reference_options(name,flows[name],calibrations,pairs) for name in queries}\n    cache=foreground_pairs(images,queries,training,out/\'foreground_scene_registrations.json\')\n    foreground={name:foreground_options(name,flows[name],calibrations,cache) for name in queries}\n    rows=[];predictions={};physical_rows=[]\n    all_versions=[\'frozen_mercedes_baseline\']+(CANDIDATES if args.phase==\'development\' else [config[\'selected_method\']])\n    for name in queries:\n        baseline=predict_hybrid(name,raw,mercedes_labels,options,\'top3weighted\',.75)\n        results={\'frozen_mercedes_baseline\':baseline}\n        for method in all_versions[1:]:results[method]=candidate(name,method,raw,flows,training,options,foreground,cache,baseline)\n        for result in results.values():\n            assert name not in result.get(\'reference_videos\',[]),\'A test became its own reference.\'\n            assert not set(plan[\'reserved_test_videos\'])&set(result.get(\'reference_videos\',[]))\n        if config and config[\'selected_method\'].startswith(\'geometry_\'):\n            for part in results[config[\'selected_method\']].get(\'geometry_calibration_components\',[]):\n                assert abs(part[\'metric_scale\']-config[\'vehicle_metric_scales\'][part[\'version\']])<1e-12\n        predictions[name]=results\n        # Test reference is read only after every prediction for that test has been produced.\n        reference=float(name.rsplit(\'_\',1)[1])\n        rows.extend(row(name,result,reference,method,args.phase) for method,result in results.items())\n        for version in sorted(raw[name]):\n            physical=predict_version(name,raw,mercedes_labels,version)\n            physical_rows.append({**row(name,physical,reference,version,args.phase),\n                                  \'raw_estimated_kmh\':raw[name][version].get(\'estimated_kmh\'),\n                                  \'raw_error_kmh\':raw[name][version].get(\'estimated_kmh\',reference)-reference if raw[name][version].get(\'available\') else None})\n        print(name,{method:round(result[\'estimated_kmh\'],3) if result.get(\'available\') else None for method,result in results.items()},flush=True)\n    summary={method:metrics([r for r in rows if r[\'version\']==method]) for method in all_versions}\n    write(out/\'results.json\',rows);export_rows(out/\'results.csv\',rows)\n    write(out/\'predictions.json\',predictions);write(out/\'metrics.json\',summary)\n    write(out/\'physical_version_results.json\',physical_rows);export_rows(out/\'physical_version_results.csv\',physical_rows)\n    write(out/\'inference_locations.json\',inference)\n    write(out/\'protocol.json\',{\'phase\':args.phase,\'calibration_references\':sorted(training),\n          \'reserved_test_references_used_for_training\':False,\'own_reference_exclusion_verified\':True,\n          \'query_to_reference_registration_speed_blind\':True,\'source_sha256\':source_hashes(),\n          \'vehicle_length_prior_m\':KIA_LENGTH_PRIOR_M,\n          \'vehicle_prior_source\':\'https://www.kia.com/content/dam/kwcms/kme/it/it/assets/contents/utility/price-list/2021/03/Listino_Kia_Sportage_Pubblico_01_03_2021.pdf\',\n          \'vehicle_prior_scope\':\'Approximate 2021 European Sportage specification; exact trim unconfirmed.\'})\n    if args.phase==\'development\':\n        selected=min(CANDIDATES,key=lambda method:sum(r[\'absolute_error_kmh\'] if r[\'absolute_error_kmh\'] is not None else 50\n                      for r in rows if r[\'version\']==method)/len(queries))\n        config={\'selected_method\':selected,\'selection\':\'Minimum 8-clip leave-one-out development MAE; unavailable prediction costs 50 km/h.\',\n                \'frozen_before_reserved_errors\':True,\'calibration_kia_videos\':plan[\'development_videos\'],\n                \'calibration_mercedes_videos\':sorted(mercedes_labels),\'development_metrics\':summary,\n                \'source_sha256\':source_hashes(),\'baseline_commit\':plan[\'frozen_baseline\'][\'source_commit\'],\n                \'scope\':\'Reference-assisted camera calibration. Eight known-speed Kia passes are extra calibration information.\',\n                \'reserved_test_count\':25}\n        config[\'vehicle_metric_scales\']={version:float(np.median([development_labels[name]/raw[name][version][\'estimated_kmh\']\n             for name in development_labels if raw[name].get(version,{}).get(\'available\')]))\n             for version in [\'v3b_rigid_distortion\',\'suv_rigid_radial\',\'v2b_delay_2d\']}\n        config[\'reference_measurement_sha256\']={name:hashlib.sha256(json.dumps(raw[name],sort_keys=True).encode()).hexdigest()\n                                                for name in development_labels}\n        write(config_path,config)\n        print(\'FROZEN METHOD:\',selected,flush=True)\n    else:\n        write(out/\'frozen_configuration_used.json\',config)\n    print(json.dumps(summary,indent=2),flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--phase\',choices=[\'development\',\'reserved\'],required=True)\n    parser.add_argument(\'--development-measurements\',required=True)\n    parser.add_argument(\'--reserved-measurements\')\n    parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--configuration\',required=True)\n    run(parser.parse_args())\n', 'render_comparison.py': '"""Render separate longitudinal-calibration and decimal-speed review videos."""\nfrom pathlib import Path\nimport json,subprocess\nimport cv2\nimport numpy as np\nfrom calibrate_speed import project\n\n\ndef distance_marks(image,result,flow,scale):\n    rows=flow.get(\'measurements\',[])\n    if len(rows)<8:return\n    points=np.array([r[\'anchor\'] for r in rows]);times=np.array([r[\'time_s\'] for r in rows])\n    if result.get(\'calibration_reused\'):\n        components=result.get(\'distance_calibration_components\',[{\'weight\':1,\n            \'query_to_reference\':result[\'query_to_reference\'],\'calibration\':result[\'reference_road_calibration\']}])\n        distance=np.zeros(len(points))\n        for component in components:\n            reference=project(points,np.array(component[\'query_to_reference\']))\n            calibration=component[\'calibration\'];h=calibration[\'horizon_y\']\n            coordinate=calibration[\'distance_coefficient_meter_pixels\']/(reference[:,1]-h)\n            sign=np.sign(np.median(np.diff(coordinate))) or 1\n            distance+=component[\'weight\']*sign*(coordinate-coordinate[0])\n        weight=result.get(\'library_weight\',1)\n        geometry=result.get(\'geometry_component_kmh\',result[\'estimated_kmh\'])/3.6*(times-times[0])\n        distance=weight*distance+(1-weight)*geometry\n    else:\n        distance=result[\'estimated_kmh\']/3.6*(times-times[0])\n    valid=np.isfinite(distance)&(distance>=0)&(distance<200)\n    points=points[valid]*scale;distance=distance[valid]\n    if len(points)<8:return\n    cv2.polylines(image,[points.astype(int)],False,(255,190,0),2)\n    order=np.argsort(distance);distance=distance[order];points=points[order]\n    if np.ptp(distance)<5:return\n    for meter in np.arange(0,min(distance.max(),100)+.1,5):\n        p=np.array([np.interp(meter,distance,points[:,axis]) for axis in [0,1]])\n        cv2.circle(image,tuple(p.astype(int)),4,(255,255,0),-1)\n        cv2.putText(image,f\'{meter:.0f} m\',tuple((p+[8,0]).astype(int)),cv2.FONT_HERSHEY_SIMPLEX,.48,(0,0,0),3)\n        cv2.putText(image,f\'{meter:.0f} m\',tuple((p+[8,0]).astype(int)),cv2.FONT_HERSHEY_SIMPLEX,.48,(255,255,0),1)\n\n\ndef render_pair(video,folder,inference,result,reference_kmh,flow):\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);inference=Path(inference)\n    observations=json.loads((inference/\'observations.json\').read_text())\n    report=json.loads((inference/\'geometry_run_report.json\').read_text())\n    tracks=json.loads((inference/\'tracks.json\').read_text())\n    geometry={};boxes={}\n    for r in observations:geometry.setdefault(r[\'frame_index\'],[]).append(r)\n    for r in tracks:boxes.setdefault(r[\'frame_index\'],[]).append(r)\n    cap=cv2.VideoCapture(str(video));assert cap.isOpened()\n    fps=float(cap.get(cv2.CAP_PROP_FPS));size=(1280,720)\n    scale=np.array([1280/report[\'width\'],720/report[\'height\']])\n    writers={name:cv2.VideoWriter(str(folder/(name+\'.avi\')),cv2.VideoWriter_fourcc(*\'MJPG\'),fps,size)\n             for name in [\'calibration\',\'speed\']}\n    assert all(w.isOpened() for w in writers.values())\n    interval=result[\'time_interval_s\'];speed=result[\'estimated_kmh\'];index=0;labelled=0\n    try:\n        while True:\n            ok,source=cap.read()\n            if not ok:break\n            calibration=cv2.resize(source,size);speed_image=calibration.copy()\n            distance_marks(calibration,result,flow,scale)\n            for r in geometry.get(index,[]):\n                points=np.array(r[\'keypoints\'])[report[\'ground_indices\'],:2]*scale\n                cv2.polylines(calibration,[points.astype(int)],True,(0,255,0),2)\n                for node,p in enumerate(points):\n                    cv2.circle(calibration,tuple(p.astype(int)),4,(0,255,255),-1)\n                    cv2.putText(calibration,str(node),tuple((p+[6,-6]).astype(int)),cv2.FONT_HERSHEY_SIMPLEX,.45,(0,255,255),1)\n            for r in boxes.get(index,[]):\n                points=np.array(r[\'box\']).reshape(2,2)*scale\n                active=r[\'track_id\']==result.get(\'track_id\') and interval[0]<=index/fps<=interval[1]\n                color=(0,255,255) if active else (160,160,160)\n                cv2.rectangle(speed_image,tuple(points[0].astype(int)),tuple(points[1].astype(int)),color,2)\n                if active:\n                    labelled+=1;position=(max(5,int(points[0,0])),max(98,int(points[0,1])-7))\n                    cv2.putText(speed_image,f\'{speed:.1f} km/h\',position,cv2.FONT_HERSHEY_SIMPLEX,.85,(0,0,0),4)\n                    cv2.putText(speed_image,f\'{speed:.1f} km/h\',position,cv2.FONT_HERSHEY_SIMPLEX,.85,color,2)\n            for image in [calibration,speed_image]:\n                cv2.rectangle(image,(0,0),(1280,78),(20,20,20),-1)\n                cv2.rectangle(image,(0,677),(1280,720),(20,20,20),-1)\n            cv2.putText(calibration,\'CALIBRATION REVIEW | vehicle nodes and local road distance\',(15,29),cv2.FONT_HERSHEY_SIMPLEX,.68,(0,220,255),2)\n            reused=result.get(\'calibration_reused\',False)\n            subtitle=\'Reused reference calibration + new vehicle geometry\' if reused else \'New calibration from vehicle geometry + visual motion\'\n            cv2.putText(calibration,subtitle,(15,60),cv2.FONT_HERSHEY_SIMPLEX,.58,(235,235,235),1)\n            footer=\'Other reference passes calibrate the scale. This test video is excluded.\' if reused else \'Car-size prior and constant-motion fit. Distances are local hypotheses.\'\n            cv2.putText(calibration,footer,(15,704),cv2.FONT_HERSHEY_SIMPLEX,.55,(230,230,230),1)\n            cv2.putText(speed_image,f\'Estimated pass speed: {speed:.1f} km/h\',(15,31),cv2.FONT_HERSHEY_SIMPLEX,.82,(0,255,255),2)\n            cv2.putText(speed_image,f\'Average over source t={interval[0]:.2f}-{interval[1]:.2f} s | Development test\',(15,62),cv2.FONT_HERSHEY_SIMPLEX,.57,(230,230,230),1)\n            cv2.putText(speed_image,f\'Cruise-control reference: {reference_kmh:.1f} km/h | Signed error: {speed-reference_kmh:+.1f} km/h\',(15,704),cv2.FONT_HERSHEY_SIMPLEX,.62,(230,230,230),1)\n            writers[\'calibration\'].write(calibration);writers[\'speed\'].write(speed_image)\n            if index==round(sum(interval)/2*fps):\n                cv2.imwrite(str(folder/\'calibration_example.jpg\'),calibration)\n                cv2.imwrite(str(folder/\'speed_example.jpg\'),speed_image)\n            index+=1\n    finally:\n        cap.release()\n        for writer in writers.values():writer.release()\n    for name in writers:\n        raw=folder/(name+\'.avi\')\n        subprocess.run([\'ffmpeg\',\'-y\',\'-loglevel\',\'error\',\'-i\',str(raw),\'-c:v\',\'libx264\',\'-crf\',\'25\',\n            \'-pix_fmt\',\'yuv420p\',\'-movflags\',\'+faststart\',str(folder/(name+\'.mp4\'))],check=True)\n        raw.unlink()\n    (folder/\'render_report.json\').write_text(json.dumps({\'frames\':index,\'fps\':fps,\'duration_s\':index/fps,\n        \'speed_display_decimal_places\':1,\'vehicle_label_frames\':labelled,\n        \'files\':[\'calibration.mp4\',\'speed.mp4\'],\'scope\':\'Longitudinal road-distance review; not a full road survey\'},indent=2),encoding=\'utf-8\')\n', 'render_cross_vehicle.py': '"""Paired videos showing the actual shared camera map and decimal pass speed."""\nfrom pathlib import Path\nimport json\nimport cv2\nimport numpy as np\nimport render_comparison\nfrom calibrate_speed import project\nfrom shared_camera_calibration import camera_coordinate, corrected_points\n\nORIGINAL_MARKS=render_comparison.distance_marks\n\n\ndef camera_marks(image,result,flow,scale):\n    model=result.get(\'shared_camera_model\')\n    components=result.get(\'geometry_calibration_components\')\n    if not model and not components:\n        return ORIGINAL_MARKS(image,result,flow,scale)\n    rows=flow.get(\'measurements\',[])\n    if len(rows)<8:return\n    points=np.array([r[\'anchor\'] for r in rows]);times=np.array([r[\'time_s\'] for r in rows])\n    keep=(times>=result[\'time_interval_s\'][0])&(times<=result[\'time_interval_s\'][1])\n    if components:\n        points=points[keep]\n        if len(points)<8:return\n        distance=np.zeros(len(points))\n        for component in components:\n            prediction=component[\'prediction\']\n            if \'mapping\' not in prediction:return ORIGINAL_MARKS(image,result,flow,scale)\n            f=prediction[\'focal_length_pixels\']\n            K=np.array([[f,0,960],[0,f,540],[0,0,1]],float)\n            distortion=np.array([prediction.get(\'radial_k1\',0),0,0,0,0],float)\n            undistorted=cv2.undistortPoints(points[:,None,:].astype(float),K,distortion,P=K)[:,0,:]\n            world=project(undistorted,np.asarray(prediction[\'mapping\']))*component[\'metric_scale\']\n            heading=world[-1]-world[0];heading/=max(np.linalg.norm(heading),1e-9)\n            distance+=component[\'weight\']*((world-world[0])@heading)\n    else:\n        aligned=project(points,np.asarray(result[\'query_to_template\']))\n        y=corrected_points(aligned,model.get(\'radial_lambda\',0))[:,1]\n        low,high=model[\'support_y_pixels\'];keep&=(y>=low)&(y<=high)\n        points=points[keep];aligned=aligned[keep]\n        if len(points)<8:return\n        coordinate=camera_coordinate(aligned,model)\n        sign=np.sign(np.median(np.diff(coordinate))) or 1\n        distance=sign*(coordinate-coordinate[0])\n    points=points*scale\n    valid=np.isfinite(distance)&(distance>=0)&(distance<200)\n    points=points[valid];distance=distance[valid]\n    if len(points)<8:return\n    cv2.polylines(image,[points.astype(int)],False,(255,190,0),2)\n    order=np.argsort(distance);distance=distance[order];points=points[order]\n    for meter in np.arange(0,min(distance.max(),100)+.1,5):\n        p=np.array([np.interp(meter,distance,points[:,axis]) for axis in [0,1]])\n        cv2.circle(image,tuple(p.astype(int)),4,(255,255,0),-1)\n        cv2.putText(image,f\'{meter:.0f} m\',tuple((p+[8,0]).astype(int)),cv2.FONT_HERSHEY_SIMPLEX,.48,(0,0,0),3)\n        cv2.putText(image,f\'{meter:.0f} m\',tuple((p+[8,0]).astype(int)),cv2.FONT_HERSHEY_SIMPLEX,.48,(255,255,0),1)\n\n\ndef render(video,folder,inference,result,reference_kmh,flow,phase):\n    render_comparison.distance_marks=camera_marks\n    try:render_comparison.render_pair(video,folder,inference,result,reference_kmh,flow)\n    finally:render_comparison.distance_marks=ORIGINAL_MARKS\n    path=Path(folder)/\'render_report.json\'\n    report=json.loads(path.read_text(encoding=\'utf-8\'))\n    report.update(phase=phase,calibration_method=result.get(\'adaptation\',result.get(\'method\')),\n                  reference_videos=result.get(\'reference_videos\',[]),\n                  own_reference_speed_used=False,\n                  distance_marks_use_fitted_shared_camera=bool(result.get(\'shared_camera_model\')))\n    report[\'distance_marks_use_fitted_vehicle_camera\']=bool(result.get(\'geometry_calibration_components\'))\n    path.write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n', 'publish_kia_report.py': '"""Package individual Kia tests, before/after evidence and paired output videos."""\nimport argparse\nimport hashlib\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom calibration_comparison import metrics\nfrom kia_validation import read,write,export_rows\n\n\ndef bands(rows):\n    return {label:metrics([r for r in rows if low<=r[\'reference_kmh\']<high])\n            for label,low,high in [(\'below_50\',0,50),(\'50_to_79\',50,80),(\'80_plus\',80,200)]}\n\n\ndef package(args):\n    root=Path(__file__).resolve().parent;out=Path(args.output);out.mkdir(parents=True,exist_ok=True)\n    dev=Path(args.development_evaluation);test=Path(args.reserved_evaluation)\n    configuration=read(args.configuration);method=configuration[\'selected_method\']\n    assert read(test/\'frozen_configuration_used.json\')==configuration\n    all_rows=read(dev/\'results.json\')+read(test/\'results.json\')\n    baseline=[r for r in all_rows if r[\'version\']==\'frozen_mercedes_baseline\']\n    improved=[r for r in all_rows if r[\'version\']==method]\n    assert len(improved)==len(baseline)==33\n    baseline_by_name={r[\'video\']:r for r in baseline}\n    for row in improved:\n        before=baseline_by_name[row[\'video\']]\n        row.update(previous_estimated_kmh=before[\'estimated_kmh\'],previous_error_kmh=before[\'error_kmh\'])\n    summary={phase:{\'before\':metrics([r for r in baseline if phase==\'all_33\' or r[\'phase\']==phase]),\n                    \'after\':metrics([r for r in improved if phase==\'all_33\' or r[\'phase\']==phase])}\n             for phase in [\'development\',\'reserved\',\'all_33\']}\n    summary[\'reserved_accuracy_target_met\']=summary[\'reserved\'][\'after\'][\'within_2_kmh\']==25\n    pattern={\'before_by_speed\':bands(baseline),\'after_by_speed\':bands(improved),\n             \'speed_absolute_error_correlation_before\':float(np.corrcoef([r[\'reference_kmh\'] for r in baseline],\n                 [r[\'absolute_error_kmh\'] for r in baseline])[0,1]),\n             \'speed_absolute_error_correlation_after\':float(np.corrcoef([r[\'reference_kmh\'] for r in improved],\n                 [r[\'absolute_error_kmh\'] for r in improved])[0,1]),\n             \'scope\':\'Descriptive diagnostics only. Reserved errors were not used to choose the frozen method.\'}\n    for phase,source in [(\'development\',dev),(\'reserved\',test)]:shutil.copytree(source,out/phase,dirs_exist_ok=True)\n    write(out/\'configuration.json\',configuration);write(out/\'metrics.json\',summary)\n    write(out/\'patterns.json\',pattern);write(out/\'results.json\',improved);export_rows(out/\'results.csv\',improved)\n    for phase,measurements in [(\'development\',args.development_measurements),(\'reserved\',args.reserved_measurements)]:\n        shutil.copytree(measurements,out/\'measurements\'/phase,dirs_exist_ok=True)\n    if args.mercedes_comparison:\n        shutil.copytree(args.mercedes_comparison,out/\'mercedes_regression\',dirs_exist_ok=True)\n    fig,axes=plt.subplots(1,2,figsize=(13,5))\n    speeds=np.array([r[\'reference_kmh\'] for r in improved]);order=np.argsort(speeds)\n    axes[0].plot([25,110],[25,110],color=\'gray\',linestyle=\'--\',label=\'Reference\')\n    axes[0].scatter(speeds,[r[\'estimated_kmh\'] for r in baseline],label=\'Frozen Mercedes transfer\',s=28)\n    axes[0].scatter(speeds,[r[\'estimated_kmh\'] for r in improved],label=\'Adapted camera calibration\',s=28)\n    axes[0].set(xlabel=\'Cruise-control reference (km/h)\',ylabel=\'Estimated speed (km/h)\',title=\'All 33 Kia clips\')\n    axes[0].legend(fontsize=8)\n    for rows,label in [(baseline,\'Before\'),(improved,\'After\')]:\n        axes[1].plot(speeds[order],np.array([r[\'absolute_error_kmh\'] for r in rows])[order],\'.-\',label=label)\n    axes[1].axhline(2,color=\'gray\',linestyle=\'--\',label=\'2 km/h tolerance\')\n    axes[1].set(xlabel=\'Reference (km/h)\',ylabel=\'Absolute error (km/h)\',title=\'Development and reserved tests combined\')\n    axes[1].legend(fontsize=8);fig.tight_layout();fig.savefig(out/\'before_after.jpg\',dpi=150);plt.close(fig)\n    if not args.no_render:\n        from render_cross_vehicle import render\n        for row in improved:\n            name=Path(row[\'video\']).stem;phase=row[\'phase\'];measurements=out/\'measurements\'/phase\n            prediction=read(out/phase/\'predictions.json\')[name][method]\n            render(Path(args.video_dir)/row[\'video\'],out/\'videos\'/name,measurements/\'inference\'/name,\n                   prediction,row[\'reference_kmh\'],read(measurements/\'round2\'/name/\'adaptive_flow_features.json\'),phase)\n            print(\'Rendered paired videos:\',name,flush=True)\n    m=summary[\'reserved\'];d=summary[\'development\'];a=summary[\'all_33\']\n    lines=[\'# Kia Sportage: cross-vehicle calibration tests\',\'\',\n           \'Run date: 2026-10-16 (Asia/Karachi). All 33 clips are separate tests. Filename speeds are constant cruise-control references, confirmed by the user. Text annotation files are ignored.\',\'\',\n           \'**Primary result: the 25 reserved clips, evaluated after the method was frozen.**\',\'\',\n           \'| Group | Before MAE | After MAE | After within ±2 km/h | Worst after error |\',\n           \'|---|---:|---:|---:|---:|\']\n    for label,value in [(\'Eight development clips (leave-one-out calibration; used for method selection)\',d),\n                        (\'25 reserved clips (not used for method selection)\',m),(\'All 33 (mixed summary)\',a)]:\n        x=value[\'after\'];lines.append(f"| {label} | {value[\'before\'][\'mae_kmh\']:.2f} | {x[\'mae_kmh\']:.2f} | {x[\'within_2_kmh\']}/{x[\'clips\']} | {x[\'maximum_absolute_error_kmh\']:.2f} |")\n    lines+=[\'\',f"Selected method: **{method}**. The ±2 km/h target across every reserved clip is **{\'met\' if summary[\'reserved_accuracy_target_met\'] else \'not met\'}**.",\'\',\n            \'The before result transfers the frozen Mercedes configuration and its 30-pass reference library to the Kia. The adapted result can use eight additional known-speed Kia reference passes. These are different calibration inputs: improvement must not be attributed solely to a better automatic model. The reserved clips share the same location and Kia recording session; they do not establish accuracy at a new camera or site.\',\'\',\n            \'![Before and after](before_after.jpg)\',\'\',\n            \'## Protocol and versions\',\'\',\n            \'The [experiment plan](../../kia_experiment_plan.json) declared the eight development clips and 25 reserved clips before Kia errors were examined. Every development test excludes its own speed and reference calibration. The selected method and source hashes were then frozen in [configuration.json](configuration.json), before the reserved errors were calculated. No reserved speed enters a calibration fit or method selection. No neural-network weights were fine-tuned.\',\'\',\n            \'New comparisons separate a vehicle-length prior, an expanded reference library using the previous model, lower-image scene registration, shared reciprocal camera maps from optical flow, shared trajectory fitting, a lens-aware trajectory fit, and quadratic/cubic local scale curves. The second round adds per-video SUV template/lens fits and native/SUV geometry ensembles. The ensemble corrects each template’s metric scale from other known-speed Kia passes, then combines their independent per-video camera fits. This empirical template correction is vehicle-specific and is not automatic identification of the true dimensions. Unavailable shared maps fall back explicitly to the frozen baseline. Such fallbacks remain in the error totals.\',\'\',\n            \'The approximate 4.485 m Kia length comes from a [2021 Kia specification](https://www.kia.com/content/dam/kwcms/kme/it/it/assets/contents/utility/price-list/2021/03/Listino_Kia_Sportage_Pubblico_01_03_2021.pdf). The exact year/trim is unconfirmed, so this is a dimension prior, not an identified vehicle measurement. The filenames and speeds match the [VS13 paper](https://slobodan.ucg.ac.me/science/vs13/paper.pdf), which describes one Kia and three Mercedes recording sessions. That match is supporting context, not a byte-level source-identification claim.\',\'\',\n            \'| Development version | MAE km/h | Within ±2 | Worst error |\',\'|---|---:|---:|---:|\']\n    for version,value in read(dev/\'metrics.json\').items():\n        lines.append(f"| {version} | {value[\'mae_kmh\']:.2f} | {value[\'within_2_kmh\']}/8 | {value[\'maximum_absolute_error_kmh\']:.2f} |")\n    lines+=[\'\',\'All 18 previous physical/model versions and six new geometry ablations are retained per clip in each group’s `physical_version_results.csv`. Previous versions show their Mercedes-transferred scale; new ablations retain raw values and have no Mercedes scale if no corresponding pilot exists. These diagnostics were not used to change the chosen method after reserved evaluation.\',\'\',\n            \'## Patterns\',\'\',\n            \'| Speed band (all 33; descriptive) | Before MAE | After MAE | After within ±2 |\',\n            \'|---|---:|---:|---:|\']\n    for label,old in pattern[\'before_by_speed\'].items():\n        new=pattern[\'after_by_speed\'][label]\n        lines.append(f"| {label} | {old[\'mae_kmh\']:.2f} | {new[\'mae_kmh\']:.2f} | {new[\'within_2_kmh\']}/{new[\'clips\']} |")\n    lines+=[\'\',\'Detailed signed errors, reference reuse, fallbacks and software/model provenance are retained. Correlation and band summaries describe patterns; they do not prove a cause.\',\'\',\n            \'## Each video and its calibration/speed outputs\',\'\',\n            \'| Clip / outputs | Group | Reference | Before | After | Signed error | Within ±2 |\',\n            \'|---|---|---:|---:|---:|---:|---|\']\n    for row in sorted(improved,key=lambda r:r[\'reference_kmh\']):\n        name=Path(row[\'video\']).stem\n        lines.append(f"| [{row[\'video\']}](videos/{name}/speed.mp4) · [calibration](videos/{name}/calibration.mp4) | {row[\'phase\']} | {row[\'reference_kmh\']:.1f} | {row[\'previous_estimated_kmh\']:.1f} | {row[\'estimated_kmh\']:.1f} | {row[\'error_kmh\']:+.2f} | {\'Yes\' if row[\'within_2_kmh\'] else \'No\'} |")\n    lines+=[\'\',\'## Evidence and limits\',\'\',\n            \'- [Full numeric results](results.csv) and [group metrics](metrics.json).\',\n            \'- [Development results](development/results.csv), [reserved results](reserved/results.csv), and their full predictions/reference lists.\',\n            \'- [Speed-band diagnostics](patterns.json), [verification](artifact_verification.json), and [provenance](provenance.json).\',\n            \'- The speed videos show an estimated constant pass speed to one decimal place. Calibration videos show predicted vehicle ground nodes and local distance marks; they are not a complete survey of the road.\',\n            \'- Cruise-control settings are the evaluation reference. Results are reference-assisted research on these recordings, with a reserved same-session check.\',\'\']\n    (out/\'README.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n    print(json.dumps(summary,indent=2),flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    for name in [\'development-evaluation\',\'reserved-evaluation\',\'development-measurements\',\'reserved-measurements\',\n                 \'configuration\',\'output\',\'video-dir\']:parser.add_argument(\'--\'+name,required=True)\n    parser.add_argument(\'--mercedes-comparison\');parser.add_argument(\'--no-render\',action=\'store_true\')\n    package(parser.parse_args())\n', 'test_shared_camera_calibration.py': '"""Physical checks for shared camera fitting and metric trajectory prediction."""\nimport unittest\nimport numpy as np\nfrom shared_camera_calibration import fit_shared_camera, predict_shared_camera\n\n\ndef synthetic_pass(mps, radial_lambda=0, noise=0, seed=17):\n    t=np.linspace(0,1,90);dt=1/30;depth=50+mps*.4-mps*t\n    def points(z):\n        xy=np.c_[5000/z,370+14000/z-540]/1920\n        if radial_lambda:\n            ru=np.linalg.norm(xy,axis=1)\n            ratio=(1-np.sqrt(1-4*radial_lambda*ru**2))/(2*radial_lambda*ru**2)\n            xy*=ratio[:,None]\n        return xy*1920+[960,540]\n    p=points(depth);q=points(depth-mps*dt)\n    if noise:\n        rng=np.random.default_rng(seed);p+=rng.normal(0,noise,p.shape);q+=rng.normal(0,noise,q.shape)\n    return {\'track_id\':1,\'measurements\':[{\'time_s\':float(time),\'dt_s\':dt,\n             \'anchor\':a.tolist(),\'transported_anchor\':b.tolist(),\'inliers\':30,\n             \'flow_rms_pixels\':.15} for time,a,b in zip(t,p,q)]}\n\n\nclass SharedCameraTests(unittest.TestCase):\n    def setUp(self):\n        self.flows={str(v):synthetic_pass(v) for v in [10,16,23,30]}\n        self.speeds={str(v):v*3.6 for v in [10,16,23,30]}\n        self.transforms={name:np.eye(3) for name in self.flows}\n\n    def test_shared_trajectory_and_flow_predict_unseen_speed(self):\n        for family in [\'trajectory\',\'flow\']:\n            with self.subTest(family=family):\n                model=fit_shared_camera(self.flows,self.speeds,self.transforms,family)\n                self.assertTrue(model[\'available\'])\n                self.assertAlmostEqual(model[\'horizon_y\'],370,places=2)\n                result=predict_shared_camera(synthetic_pass(22),model,np.eye(3))\n                self.assertTrue(result[\'available\'])\n                self.assertAlmostEqual(result[\'estimated_kmh\'],79.2,places=3)\n\n    def test_radial_trajectory_recovers_unseen_motion(self):\n        flows={str(v):synthetic_pass(v,.3) for v in [10,16,23,30]}\n        model=fit_shared_camera(flows,self.speeds,self.transforms,\'trajectory_radial\')\n        self.assertTrue(model[\'available\'])\n        result=predict_shared_camera(synthetic_pass(22,.3),model,np.eye(3))\n        self.assertLess(abs(result[\'estimated_kmh\']-79.2),.1)\n\n    def test_no_reference_support_returns_unavailable(self):\n        model=fit_shared_camera(self.flows,{\'10\':36},self.transforms)\n        self.assertFalse(model[\'available\'])\n        model=fit_shared_camera(self.flows,self.speeds,self.transforms)\n        query=synthetic_pass(22)\n        for row in query[\'measurements\']:\n            row[\'anchor\'][1]+=2000;row[\'transported_anchor\'][1]+=2000\n        self.assertFalse(predict_shared_camera(query,model,np.eye(3))[\'available\'])\n\n    def test_query_annotations_do_not_enter_prediction(self):\n        model=fit_shared_camera(self.flows,self.speeds,self.transforms)\n        query=synthetic_pass(22)\n        original=predict_shared_camera(query,model,np.eye(3))[\'estimated_kmh\']\n        query[\'filename\']=\'KiaSportage_999\';query[\'reference_speed\']=999\n        self.assertEqual(predict_shared_camera(query,model,np.eye(3))[\'estimated_kmh\'],original)\n        self.assertNotIn(\'KiaSportage_999\',model[\'reference_videos\'])\n\n    def test_outer_reference_is_removed_before_camera_fit(self):\n        from kia_validation import candidate\n        flows={**self.flows,\'query\':synthetic_pass(22)}\n        cache={\'query|\'+name:{\'accepted\':True,\'query_to_template\':np.eye(3).tolist()} for name in flows}\n        baseline={\'available\':True,\'estimated_kmh\':1}\n        original=candidate(\'query\',\'shared_trajectory\',{},flows,\n                           {**self.speeds,\'query\':79.2},{},{},cache,baseline)\n        poisoned=candidate(\'query\',\'shared_trajectory\',{},flows,\n                           {**self.speeds,\'query\':999},{},{},cache,baseline)\n        self.assertAlmostEqual(original[\'estimated_kmh\'],79.2,places=3)\n        self.assertEqual(original[\'estimated_kmh\'],poisoned[\'estimated_kmh\'])\n        self.assertNotIn(\'query\',poisoned[\'reference_videos\'])\n\n    def test_geometry_ensemble_uses_other_reference_scales_only(self):\n        from kia_validation import candidate\n        names=[\'KiaSportage_31\',\'KiaSportage_55\',\'KiaSportage_77\']\n        labels={name:float(name.rsplit(\'_\',1)[1]) for name in names}\n        def prediction(value):return {\'available\':True,\'estimated_kmh\':value,\'time_interval_s\':[0,1],\'track_id\':1}\n        raw={name:{\'v3b_rigid_distortion\':prediction(speed/1.2),\'suv_rigid_radial\':prediction(speed/1.4)}\n             for name,speed in labels.items()}\n        query=\'KiaSportage_44\'\n        raw[query]={\'v3b_rigid_distortion\':prediction(40),\'suv_rigid_radial\':prediction(35)}\n        results=[]\n        for poison in [44,999]:\n            results.append(candidate(query,\'geometry_ensemble_50\',raw,{},\n                                     {**labels,query:poison},{},{},{},{\'available\':True,\'estimated_kmh\':1}))\n        self.assertAlmostEqual(results[0][\'estimated_kmh\'],48.5)\n        self.assertEqual(results[0][\'estimated_kmh\'],results[1][\'estimated_kmh\'])\n        self.assertEqual(set(results[0][\'reference_videos\']),set(labels))\n\n\nif __name__==\'__main__\':unittest.main()\n', 'test_calibration_candidates.py': '"""Known-geometry and outlier checks for the calibration comparison."""\nimport unittest\nimport cv2\nimport numpy as np\nfrom calibration_candidates import temporal_delay_2d,rigid_cuboid_bundle,vp_rigid_bundle\nfrom flow_calibration import estimate_flow\nfrom calibrate_speed import project\nfrom lens_calibration import fit_motion_lens,distort_division\nfrom longitudinal_calibration import fit_longitudinal_reference,predict_longitudinal\nfrom calibration_comparison import select_fold,PILOTS\n\n\nclass CandidateTests(unittest.TestCase):\n    def test_temporal_delay_recovers_metric_speed_under_perspective(self):\n        camera=np.array([[1000,960,0],[0,540,1200],[0,1,0]],float)\n        rows=[]\n        for t in np.linspace(0,2,40):\n            y=12+4*t\n            q=np.array([[1-.9,y-2.45],[1-.9,y+2.45],[1+.9,y+2.45],[1+.9,y-2.45]])\n            rows.append({\'quad\':project(q,camera).tolist(),\'time_s\':float(t),\'track_id\':1})\n        result=temporal_delay_2d(rows)\n        self.assertTrue(result[\'available\'])\n        self.assertAlmostEqual(result[\'estimated_kmh\'],14.4,delta=.1)\n\n    def test_rigid_bundle_recovers_constant_speed_from_known_3d_projection(self):\n        L,W,H=4.9,1.87,1.45\n        base=np.array([[-W/2,0,0],[-W/2,0,L],[W/2,0,L],[W/2,0,0]])\n        points=np.r_[base,base+[0,-H,0]]\n        R=cv2.Rodrigues(np.array([.05,.3,.01]))[0];observations=[]\n        for index,t in enumerate(np.linspace(0,1,32)):\n            XYZ=points@R.T+np.array([1,1.2,15])-(t-.5)*12*R[:,2]\n            xy=XYZ[:,:2]/XYZ[:,2:]*1400+[960,540]\n            box=np.r_[xy.min(axis=0),xy.max(axis=0)].tolist()\n            observations.append({\'track_id\':1,\'time_s\':float(t),\'frame_index\':index,\n                \'baseline_box\':box,\'keypoints\':xy.tolist(),\'geometry_confidence\':.9,\'association_iou\':.9})\n        report={\'width\':1920,\'height\':1080,\'ground_indices\':[0,1,2,3]}\n        result=rigid_cuboid_bundle(observations,report)\n        self.assertTrue(result[\'available\'])\n        self.assertAlmostEqual(result[\'estimated_kmh\'],43.2,delta=.5)\n        renamed=rigid_cuboid_bundle(observations,{**report,\'source_video\':\'MercedesAMG550_999.MP4\'})\n        self.assertAlmostEqual(renamed[\'estimated_kmh\'],result[\'estimated_kmh\'],places=6)\n        direction=R[:,2];vp=direction[:2]/direction[2]*1400+[960,540]\n        constrained=vp_rigid_bundle(observations,report,vp)\n        self.assertAlmostEqual(constrained[\'estimated_kmh\'],43.2,delta=.5)\n\n    def test_flow_uses_actual_time_and_rejects_large_motion_outliers(self):\n        mapping=np.diag([.02,.02,1]);measurements=[]\n        for i in range(40):\n            dt=1/29.97;anchor=np.array([300+i*3,600+i])\n            velocity=np.array([10.,0.])\n            if i%7==0:velocity*=8\n            transported=anchor+velocity/.02*dt\n            measurements.append({\'time_s\':i*dt,\'dt_s\':dt,\'anchor\':anchor.tolist(),\n                                  \'transported_anchor\':transported.tolist()})\n        result=estimate_flow({\'track_id\':1,\'measurements\':measurements},mapping)\n        self.assertTrue(result[\'available\'])\n        self.assertAlmostEqual(result[\'estimated_kmh\'],36,places=6)\n        self.assertLess(result[\'observations\'],len(measurements))\n\n    def test_reference_road_calibration_predicts_a_different_speed(self):\n        def make_flow(speed):\n            rows=[];vp=np.array([960.,540.])\n            for index,t in enumerate(np.linspace(0,1,40)):\n                dt=1/29.97;z=12+speed*t\n                point=vp+np.array([1000.,1200.])/z\n                after=vp+np.array([1000.,1200.])/(z+speed*dt)\n                scale=z/(z+speed*dt);A=np.c_[np.eye(2)*scale,(1-scale)*vp]\n                rows.append({\'time_s\':float(t),\'dt_s\':dt,\'anchor\':point.tolist(),\n                    \'transported_anchor\':after.tolist(),\'affine\':A.tolist(),\'crop_origin\':[0,0],\'inliers\':20})\n            return {\'track_id\':1,\'measurements\':rows}\n        calibration=fit_longitudinal_reference(make_flow(10),36)\n        prediction=predict_longitudinal(make_flow(20),calibration,np.eye(3))\n        self.assertTrue(prediction[\'available\'])\n        self.assertAlmostEqual(prediction[\'estimated_kmh\'],72,delta=.1)\n\n    def test_motion_lens_fit_recovers_known_radial_distortion_without_speed_labels(self):\n        rng=np.random.default_rng(12);vp=np.array([-.25,.01]);rows=[]\n        for i in range(20):\n            a=rng.uniform([-.1,.03],[.4,.24],size=(30,2));b=vp+(a-vp)*(1.03+i*.001)\n            p=distort_division(a,.3,[0,0],1)*1920+[960,540]\n            q=distort_division(b,.3,[0,0],1)*1920+[960,540]\n            A=np.c_[np.eye(2)*1.03,-.03*(vp*1920+[960,540])]\n            rows.append({\'matched_points_image\':p.tolist(),\'matched_next_points_image\':q.tolist(),\n                         \'affine\':A.tolist(),\'crop_origin\':[0,0],\'inliers\':30})\n        result=fit_motion_lens({\'measurements\':rows},1920,1080)\n        self.assertTrue(result[\'available\'])\n        self.assertAlmostEqual(result[\'division_coefficient\'],.3,delta=.02)\n\n    def test_outer_test_label_cannot_change_calibration_or_method_selection(self):\n        names=PILOTS+[\'anonymous_test\'];labels=dict(zip(names,[30.,60.,100.,80.]))\n        raw={name:{\'v2d_flow_vp_delay\':{\'available\':True,\'estimated_kmh\':value/1.1}}\n             for name,value in labels.items()}\n        options={name:[{\'reference\':other,\'score\':10,\'prediction\':{\'estimated_kmh\':labels[name],\n                        \'query_to_reference\':np.eye(3).tolist(),\'reference_road_calibration\':{\'horizon_y\':540}},\n                        \'registration\':{}} for other in names if other!=name] for name in names}\n        first=select_fold(\'anonymous_test\',raw,labels,options)\n        import copy\n        poisoned=copy.deepcopy(options)\n        for candidates in poisoned.values():\n            for candidate in candidates:\n                if candidate[\'reference\']==\'anonymous_test\':candidate[\'prediction\'][\'estimated_kmh\']+=1000\n        second=select_fold(\'anonymous_test\',raw,{**labels,\'anonymous_test\':999},poisoned)\n        self.assertEqual(first[\'estimated_kmh\'],second[\'estimated_kmh\'])\n        self.assertEqual(first[\'selection\'],second[\'selection\'])\n        self.assertNotIn(\'anonymous_test\',first[\'reference_videos\'])\n\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', 'lens_calibration.py': '"""Division-model radial self-calibration from one rigid translation\'s optical flow.\n\nUses a standard division lens model and an independently implemented robust\nepipole fit. It is not Fitzgibbon\'s two-view quadratic-eigenvalue algorithm.\n"""\nimport numpy as np\nfrom scipy.optimize import least_squares\nfrom calibration_candidates import select_rows,shared_vp_delay\nfrom refine_flow import flow_vanishing_point\n\n\ndef undistort_division(points,coefficient,center,scale):\n    p=(np.asarray(points)-center)/scale\n    return p/(1+coefficient*np.sum(p*p,axis=-1,keepdims=True))*scale+center\n\n\ndef distort_division(points,coefficient,center,scale):\n    p=(np.asarray(points)-center)/scale;r2=np.sum(p*p,axis=-1,keepdims=True)\n    return p*2/(1+np.sqrt(np.maximum(1-4*coefficient*r2,1e-8)))*scale+center\n\n\ndef fit_motion_lens(features,width,height):\n    source=[];dest=[]\n    for row in features.get(\'measurements\',[]):\n        p=np.asarray(row.get(\'matched_points_image\',[]));q=np.asarray(row.get(\'matched_next_points_image\',[]))\n        if len(p):\n            selected=np.unique(np.linspace(0,len(p)-1,min(30,len(p))).astype(int))\n            source.extend(p[selected]);dest.extend(q[selected])\n    if len(source)<100:return {\'available\':False,\'reason\':\'insufficient_visual_correspondences\'}\n    center=np.array([width/2,height/2]);scale=max(width,height)\n    a=(np.asarray(source)-center)/scale;b=(np.asarray(dest)-center)/scale\n    weights=np.clip(np.linalg.norm(b-a,axis=1)*scale,1,20)\n    initial_vp=(np.asarray(flow_vanishing_point(features)[\'point\'])-center)/scale\n    def convert(p,k):return p/np.maximum(1+k*np.sum(p*p,axis=1,keepdims=True),.1)\n    def residual(parameters):\n        vp=parameters[:2];k=parameters[2];p=convert(a,k);q=convert(b,k)\n        ray=p-vp;motion=q-p\n        cross=ray[:,0]*motion[:,1]-ray[:,1]*motion[:,0]\n        sine=cross/(np.linalg.norm(ray,axis=1)*np.linalg.norm(motion,axis=1)+1e-12)\n        den=np.r_[1+k*np.sum(a*a,axis=1),1+k*np.sum(b*b,axis=1)]\n        return np.r_[sine*weights,np.maximum(.25-den,0)*20,parameters[2]*.01]\n    fits=[]\n    for k in [-1,0,1]:\n        fit=least_squares(residual,np.r_[initial_vp,k],bounds=([-2,-2,-2],[2,2,2]),\n                          loss=\'soft_l1\',f_scale=.25,max_nfev=300)\n        score=float(np.mean(np.minimum(residual(fit.x)**2,1)));fits.append((score,fit))\n    score,fit=min(fits,key=lambda x:x[0]);k=float(fit.x[2])\n    singular=np.linalg.svd(fit.jac,compute_uv=False)\n    condition=float(singular[0]/max(singular[-1],1e-12))\n    available=abs(k)<1.98 and condition<1e5\n    return {\'available\':bool(available),\'division_coefficient\':k,\'center\':center.tolist(),\'scale\':scale,\n        \'vanishing_point_undistorted\':(fit.x[:2]*scale+center).tolist(),\n        \'correspondences\':len(a),\'robust_normal_flow_score\':score,\'jacobian_condition\':condition,\n        \'reason\':\'estimated_from_visual_motion\' if available else \'distortion_fit_at_bound_or_ill_conditioned\',\n        \'reference_label_used\':False}\n\n\ndef distortion_aware_delay(observations,report,features):\n    lens=fit_motion_lens(features,report[\'width\'],report[\'height\'])\n    if not lens[\'available\']:return {\'available\':False,\'reason\':lens[\'reason\'],\'lens_calibration\':lens}\n    corrected=[]\n    for row in observations:\n        points=np.asarray(row[\'keypoints\']).copy()\n        points[:,:2]=undistort_division(points[:,:2],lens[\'division_coefficient\'],lens[\'center\'],lens[\'scale\'])\n        corrected.append({**row,\'keypoints\':points.tolist()})\n    result=shared_vp_delay(select_rows(corrected,report),vanishing_point=lens[\'vanishing_point_undistorted\'],vp_weight=20)\n    result.update(method=\'distortion_aware_flow_vp_temporal_calibration\',lens_calibration=lens)\n    for key in [\'fitted_front_image\',\'fitted_rear_image\']:\n        if key in result:\n            result[key]=distort_division(result[key],lens[\'division_coefficient\'],lens[\'center\'],lens[\'scale\']).tolist()\n    return result\n'}
plan={'created_date': '2026-10-16', 'timezone': 'Asia/Karachi', 'development_videos': ['KiaSportage_31', 'KiaSportage_44', 'KiaSportage_55', 'KiaSportage_65', 'KiaSportage_77', 'KiaSportage_85', 'KiaSportage_96', 'KiaSportage_105'], 'reserved_test_videos': ['KiaSportage_100', 'KiaSportage_103', 'KiaSportage_33', 'KiaSportage_35', 'KiaSportage_38', 'KiaSportage_41', 'KiaSportage_46', 'KiaSportage_48', 'KiaSportage_51', 'KiaSportage_53', 'KiaSportage_58', 'KiaSportage_61', 'KiaSportage_63', 'KiaSportage_68', 'KiaSportage_69', 'KiaSportage_72', 'KiaSportage_74', 'KiaSportage_78', 'KiaSportage_80', 'KiaSportage_83', 'KiaSportage_86', 'KiaSportage_89', 'KiaSportage_91', 'KiaSportage_93', 'KiaSportage_98'], 'split_rule': 'Eight speeds spaced across the range, chosen before Kia speed errors were calculated.', 'reserved_protocol': 'Do not inspect reserved speed errors until improved configuration is frozen. Speed-blind image measurements may be computed for all videos.', 'reference': 'Filename speed in km/h; text annotation files are ignored.', 'frozen_baseline': {'source_commit': '63c1b3923153cfaf92b1faba2292d62a8261015b', 'configuration': 'top3weighted_weight_0.75', 'reference_library': '30 Mercedes passes only; no Kia speed reference', 'configuration_sha256': '584a3b111bd5c83a53244bc86b36b86f19ee6b5d3940ea0f84ab186a0d274d14'}, 'input_videos': [{'video': 'KiaSportage_100.MP4', 'bytes': 37362924, 'sha256': '236d9a2a3eb5d09c9f31800a805ab2101cd1ff10addc6247d853935f96825d95'}, {'video': 'KiaSportage_103.MP4', 'bytes': 35575998, 'sha256': '976d6e1dee24973487ff66632e84e40e8724d69f6623be4ebbb6e026ebd8b485'}, {'video': 'KiaSportage_105.MP4', 'bytes': 35178865, 'sha256': 'a1b851417da46480dfc61cfff4eec7777b84ed65e724168e3139d650df2a76db'}, {'video': 'KiaSportage_31.MP4', 'bytes': 35566715, 'sha256': 'ae8743cc9de4a9198408ce59ea8b6a61aa6d63ce168f65d97226598657d80460'}, {'video': 'KiaSportage_33.MP4', 'bytes': 37052343, 'sha256': '58e75f626fb0e1d55a28928d4ca2de1faba02f0e0a5291b1ca43fd4705d47e7d'}, {'video': 'KiaSportage_35.MP4', 'bytes': 35431463, 'sha256': '2aac26b113ede8af2414535fa58cf857f55bd9e246f646dc37e0e52286f475d7'}, {'video': 'KiaSportage_38.MP4', 'bytes': 36408043, 'sha256': 'e87ff95c831fef96035ca5ce8299e8548ed05fc584bc61ce7a2855d763397eb1'}, {'video': 'KiaSportage_41.MP4', 'bytes': 37273620, 'sha256': 'c218a59279f70717fb03371888f6b2d8ffa6c54e4227dad64c109d7c6ccdaad3'}, {'video': 'KiaSportage_44.MP4', 'bytes': 35140844, 'sha256': 'b84a23fa0b485ff286f2c6ffc0f1885e8787734d0afc64f3730ea0237d09736e'}, {'video': 'KiaSportage_46.MP4', 'bytes': 35489001, 'sha256': '1c68edf38588702d0125dc603c194213e4c254070af14871f8b7fed6487aaf50'}, {'video': 'KiaSportage_48.MP4', 'bytes': 35569426, 'sha256': '9bb55b60459519f87e5ec24d065842bfb24d5653dcfd47949855398511d21af7'}, {'video': 'KiaSportage_51.MP4', 'bytes': 36851947, 'sha256': '7e0b465f304f52467293ca5f8421038263e011a88fd1c27728d5154910c870e5'}, {'video': 'KiaSportage_53.MP4', 'bytes': 36410975, 'sha256': '80df7df8088fcfaa6bfb25fea7eddf63f18fd202e7105a39b8c7ba1aa68f214e'}, {'video': 'KiaSportage_55.MP4', 'bytes': 35226598, 'sha256': '92e2396f9ad3988c1461d5b615b3f3dd38395c0655e8fed8a64dce288c6b5416'}, {'video': 'KiaSportage_58.MP4', 'bytes': 37326499, 'sha256': '2ad445d1b7e44b9bb46e9bba8f35fb870f9de8cec6612b371b5651bfc988b1e3'}, {'video': 'KiaSportage_61.MP4', 'bytes': 35796014, 'sha256': '2c84c37fc4241b186b22d634f508785c8266ad6b988adec28af57e0e30ef3d85'}, {'video': 'KiaSportage_63.MP4', 'bytes': 37572207, 'sha256': '9c43b1b1673e92b91fdd13278c541e6c345cbe14d25d16db247439952495d607'}, {'video': 'KiaSportage_65.MP4', 'bytes': 36826485, 'sha256': 'e5a18e8f5aaa63372aca05704e0e9a763ed52dc99ee211afc9574fa8c82d0768'}, {'video': 'KiaSportage_68.MP4', 'bytes': 35497343, 'sha256': '79604d6743ec8b3a582397109a3298d256d74c2bcd03919ed91cf20ff8a07dbc'}, {'video': 'KiaSportage_69.MP4', 'bytes': 37135758, 'sha256': '8e875db26cecb8d386cdae5c8c66c7ab19e732563c05643063b05196e5db03b6'}, {'video': 'KiaSportage_72.MP4', 'bytes': 35264987, 'sha256': '7623d1c19a0d6951b22dc094d188df557a164e3a04b5d2a7e62b8b644cd62d7e'}, {'video': 'KiaSportage_74.MP4', 'bytes': 36929540, 'sha256': '8e541ebe5138211ba590ebd145e33a1feeae29dae55030b9d897ad0f2725b946'}, {'video': 'KiaSportage_77.MP4', 'bytes': 37664697, 'sha256': '41f1fbf9eeb98203b8553e4871881330852d8f7c20af70ffaad65fbfb354ba29'}, {'video': 'KiaSportage_78.MP4', 'bytes': 37290235, 'sha256': '58aeb6f11d58fbe5d0918a495680afde39c7775b797e44d11a5cac7e70d86057'}, {'video': 'KiaSportage_80.MP4', 'bytes': 34838399, 'sha256': '02b7a1b4231d4b8e783f6f239ffacb792b97b2c42fa6bd4c9ee4146b6ce0ca28'}, {'video': 'KiaSportage_83.MP4', 'bytes': 35417605, 'sha256': 'e33d089a9d7701c3dd88246d14c99f3bdc6f2e82b23b31d265ce1fe237b63137'}, {'video': 'KiaSportage_85.MP4', 'bytes': 34336769, 'sha256': '76161870e4bf414384901838daa5c4d47fb55c1dbd06d4c584d0428a2dd2b4b3'}, {'video': 'KiaSportage_86.MP4', 'bytes': 36822725, 'sha256': 'e9267b5d0c640b6de449a6867b7233019f3287a12d5cf5483c08fb3d7cb31488'}, {'video': 'KiaSportage_89.MP4', 'bytes': 37534143, 'sha256': 'bd9e3f93cbb989bb8035369c157935d0d4bf542cc3f8e29ff67625f7590b104b'}, {'video': 'KiaSportage_91.MP4', 'bytes': 36377838, 'sha256': '874104550968cbba25d8ceb628af397c48f225bd10c344de3985b19f6dd97bbb'}, {'video': 'KiaSportage_93.MP4', 'bytes': 37358794, 'sha256': 'e72585081e0a8dbeab3e8390044c38e239cb121f334ea58ada39aeb25df67b49'}, {'video': 'KiaSportage_96.MP4', 'bytes': 36946423, 'sha256': '95a491b9d75c5dfd1db2b731ae52a518c58427af04717db2725654c23fb99ae3'}, {'video': 'KiaSportage_98.MP4', 'bytes': 34610780, 'sha256': '2471f9cb545a57ff2fd7d653bd03527046f938d25a6fa9a0a84827f037f05697'}], 'scope': 'New vehicle on the same site. Reserved clips share one recording session; this is not an independent camera/site benchmark.'}
configuration={'selected_method': 'geometry_ensemble_50', 'selection': 'Minimum 8-clip leave-one-out development MAE; unavailable prediction costs 50 km/h.', 'frozen_before_reserved_errors': True, 'calibration_kia_videos': ['KiaSportage_31', 'KiaSportage_44', 'KiaSportage_55', 'KiaSportage_65', 'KiaSportage_77', 'KiaSportage_85', 'KiaSportage_96', 'KiaSportage_105'], 'calibration_mercedes_videos': ['MercedesAMG550_100', 'MercedesAMG550_105', 'MercedesAMG550_30', 'MercedesAMG550_33', 'MercedesAMG550_35', 'MercedesAMG550_38', 'MercedesAMG550_40', 'MercedesAMG550_42', 'MercedesAMG550_45', 'MercedesAMG550_47', 'MercedesAMG550_50', 'MercedesAMG550_52', 'MercedesAMG550_55', 'MercedesAMG550_58', 'MercedesAMG550_60', 'MercedesAMG550_62', 'MercedesAMG550_65', 'MercedesAMG550_67', 'MercedesAMG550_70', 'MercedesAMG550_73', 'MercedesAMG550_75', 'MercedesAMG550_78', 'MercedesAMG550_80', 'MercedesAMG550_82', 'MercedesAMG550_85', 'MercedesAMG550_87', 'MercedesAMG550_90', 'MercedesAMG550_93', 'MercedesAMG550_95', 'MercedesAMG550_98'], 'development_metrics': {'frozen_mercedes_baseline': {'clips': 8, 'estimated': 8, 'mae_kmh': 6.975414560874187, 'rmse_kmh': 11.865411552070801, 'bias_kmh': 6.679579022230742, 'maximum_absolute_error_kmh': 31.651382515028672, 'within_2_kmh': 2, 'errors_over_10_kmh': 1}, 'vehicle_length_prior': {'clips': 8, 'estimated': 8, 'mae_kmh': 5.0567515275261785, 'rmse_kmh': 7.712625876205506, 'bias_kmh': 0.20646161524589157, 'maximum_absolute_error_kmh': 20.07784705712318, 'within_2_kmh': 2, 'errors_over_10_kmh': 1}, 'expanded_reference_library': {'clips': 8, 'estimated': 8, 'mae_kmh': 6.975414560874187, 'rmse_kmh': 11.865411552070801, 'bias_kmh': 6.679579022230742, 'maximum_absolute_error_kmh': 31.651382515028672, 'within_2_kmh': 2, 'errors_over_10_kmh': 1}, 'foreground_reference_library': {'clips': 8, 'estimated': 8, 'mae_kmh': 6.325622990742164, 'rmse_kmh': 11.78687612324537, 'bias_kmh': 6.322284248618737, 'maximum_absolute_error_kmh': 31.651382515028672, 'within_2_kmh': 3, 'errors_over_10_kmh': 1}, 'shared_flow': {'clips': 8, 'estimated': 8, 'mae_kmh': 6.959680734731101, 'rmse_kmh': 11.86377462387806, 'bias_kmh': 6.370984552829336, 'maximum_absolute_error_kmh': 31.651382515028672, 'within_2_kmh': 2, 'errors_over_10_kmh': 1}, 'shared_trajectory': {'clips': 8, 'estimated': 8, 'mae_kmh': 6.928647100789457, 'rmse_kmh': 11.861034726265142, 'bias_kmh': 6.402018186770981, 'maximum_absolute_error_kmh': 31.651382515028672, 'within_2_kmh': 2, 'errors_over_10_kmh': 1}, 'shared_trajectory_radial': {'clips': 8, 'estimated': 8, 'mae_kmh': 6.975414560874187, 'rmse_kmh': 11.865411552070801, 'bias_kmh': 6.679579022230742, 'maximum_absolute_error_kmh': 31.651382515028672, 'within_2_kmh': 2, 'errors_over_10_kmh': 1}, 'shared_curve2': {'clips': 8, 'estimated': 8, 'mae_kmh': 7.077294420537648, 'rmse_kmh': 11.880040720314694, 'bias_kmh': 6.25337086702279, 'maximum_absolute_error_kmh': 31.651382515028672, 'within_2_kmh': 1, 'errors_over_10_kmh': 1}, 'shared_curve3': {'clips': 8, 'estimated': 8, 'mae_kmh': 7.071377773406574, 'rmse_kmh': 11.879000441389685, 'bias_kmh': 6.259287514153863, 'maximum_absolute_error_kmh': 31.651382515028672, 'within_2_kmh': 1, 'errors_over_10_kmh': 1}, 'geometry_native': {'clips': 8, 'estimated': 8, 'mae_kmh': 1.9731724082878572, 'rmse_kmh': 2.372117696441917, 'bias_kmh': 0.2757789509872679, 'maximum_absolute_error_kmh': 4.915829632355312, 'within_2_kmh': 5, 'errors_over_10_kmh': 0}, 'geometry_suv': {'clips': 8, 'estimated': 8, 'mae_kmh': 1.7126907716453101, 'rmse_kmh': 2.744414457928123, 'bias_kmh': -1.0103510963502527, 'maximum_absolute_error_kmh': 7.1080347812903, 'within_2_kmh': 6, 'errors_over_10_kmh': 0}, 'geometry_temporal': {'clips': 8, 'estimated': 8, 'mae_kmh': 3.248493736369404, 'rmse_kmh': 3.750136785614191, 'bias_kmh': 0.9930783211483836, 'maximum_absolute_error_kmh': 6.905008854851246, 'within_2_kmh': 2, 'errors_over_10_kmh': 0}, 'geometry_ensemble_25': {'clips': 8, 'estimated': 8, 'mae_kmh': 1.390126007706014, 'rmse_kmh': 1.8673692988977464, 'bias_kmh': -0.6888185845158699, 'maximum_absolute_error_kmh': 4.10206867787889, 'within_2_kmh': 6, 'errors_over_10_kmh': 0}, 'geometry_ensemble_50': {'clips': 8, 'estimated': 8, 'mae_kmh': 1.1748223385370196, 'rmse_kmh': 1.3495712283811419, 'bias_kmh': -0.3672860726814928, 'maximum_absolute_error_kmh': 2.66687772163737, 'within_2_kmh': 7, 'errors_over_10_kmh': 0}, 'geometry_ensemble_75': {'clips': 8, 'estimated': 8, 'mae_kmh': 1.4369845516040014, 'rmse_kmh': 1.5920533668535932, 'bias_kmh': -0.04575356084711313, 'maximum_absolute_error_kmh': 2.7484808415475115, 'within_2_kmh': 7, 'errors_over_10_kmh': 0}}, 'source_sha256': {'kia_validation.py': '3f179d0b79cf756c44f9fc601eadfeb1d43862bae7a8c7e667b2b2150c576c72', 'shared_camera_calibration.py': '24145e4157dced19899068901890e19a156de7e1b465532cd6bfdd452137a4c8', 'calibration_comparison.py': '6397a6acea08b5b0ae470f769d5fba7bf1ba482de6d3b101ee27ab10535d8e3f', 'longitudinal_calibration.py': '4d8c16217ebee6999ccfe8d27d1917cf936619a190241abc34fbe04a7598a3ec', 'scene_registration.py': '54a175367a0bfd0a3d0c83427e60821492ed7e44c84069b8cfb4e28230e023c3', 'refine_flow.py': 'c0d39acca767ba5934b815a7f1d5a36ea57d64bb931c017851c84757965c8ac0', 'calibration_candidates.py': '9659552718b2b483560eea2b8d034bd2d1f424c2435693a717d927fbeecc7931', 'vehicle_geometry_refinement.py': '120713e470d6d1aab026417450675cbcc39fff2b1a57e4273b12caf68538536a'}, 'baseline_commit': '63c1b3923153cfaf92b1faba2292d62a8261015b', 'scope': 'Reference-assisted camera calibration. Eight known-speed Kia passes are extra calibration information.', 'reserved_test_count': 25, 'vehicle_metric_scales': {'v3b_rigid_distortion': 1.06785441741998, 'suv_rigid_radial': 1.1375968583543545, 'v2b_delay_2d': 1.0431941294304905}, 'reference_measurement_sha256': {'KiaSportage_31': '3f16f8d7298639e709d0178c72c3a29dc80d626005fd515521cd72977a3e0fa3', 'KiaSportage_44': '39c057527b0a7de54e46a299a276311a6870f64cd79b1f988283607e2cfc546a', 'KiaSportage_55': 'd378862a9a0474e23a1e18c778ab9cf0be46e5274f249ebf222d0c6dab7c3eec', 'KiaSportage_65': '263cfcdd71327964b8e9b7e987c2bc699933be66374992849225e8f888958fee', 'KiaSportage_77': 'f275e215ab69178c51c9acbdebefda39dba65b7abb0a1d99caf4baa8ece54dfb', 'KiaSportage_85': 'e78816ff0de2bdf3b073b9fade9b7b260528e07b5ee971718397b2a0fab8128b', 'KiaSportage_96': '72788035b53e18b790e49db5fe601dbadfcd975b048feb980c7a7c57b1fbd524', 'KiaSportage_105': 'ce9bd5b1aba62938787f8c2f0755d59e0aba788f803ac233749bc9c6bf93a057'}}
root=Path('/kaggle/working/pipeline');root.mkdir(exist_ok=True)
for name,source in sources.items():(root/name).write_text(source,encoding='utf-8')
def write(p,v):p.parent.mkdir(parents=True,exist_ok=True);p.write_text(json.dumps(v,indent=2),encoding='utf-8')
write(root/'kia_experiment_plan.json',plan);write(root/'kia_calibration_configuration.json',configuration)
input_root=Path('/kaggle/input')
def unique(pattern):
    values=list(input_root.rglob(pattern));assert len(values)==1,(pattern,values);return values[0]
mercedes=unique('calibration_comparison/raw_predictions.json').parent
old_inference=unique('validation_inference/MercedesAMG550_30/observations.json').parent.parent
(root/'output').mkdir(exist_ok=True)
(root/'output/calibration_comparison').symlink_to(mercedes,target_is_directory=True)
(root/'output/mercedes_validation').mkdir(exist_ok=True)
(root/'output/mercedes_validation/inference').symlink_to(old_inference,target_is_directory=True)
cache=unique('cache_manifest.json').parent
work=Path('/kaggle/working/evaluation');work.mkdir(exist_ok=True)
measurements={}
for phase,first in [('development','KiaSportage_31'),('reserved','KiaSportage_33')]:
    original=unique('kia_measurements/inference/'+first+'/observations.json').parent.parent.parent
    target=work/('measurements_'+phase);target.mkdir(exist_ok=True)
    shutil.copytree(original/'inference',target/'inference',ignore=shutil.ignore_patterns('geometry_*.jpg'))
    for directory in ['round1','round2']:shutil.copytree(original/directory,target/directory)
    shutil.copytree(cache/'geometry_refinement'/phase,target/'geometry_refinement')
    shutil.copytree(cache/(phase+'_evaluation'),work/(phase+'_evaluation'))
    measurements[phase]=target
def run(script,*args):subprocess.check_call([sys.executable,str(root/script),*map(str,args)],cwd=root)
subprocess.check_call([sys.executable,'-m','unittest','test_shared_camera_calibration','test_calibration_candidates','-v'],cwd=root)
run('kia_validation.py','--phase','development','--development-measurements',measurements['development'],
    '--output',work/'development_evaluation','--configuration',work/'recomputed_configuration.json')
recomputed=json.loads((work/'recomputed_configuration.json').read_text())
for key in ['selected_method','vehicle_metric_scales','reference_measurement_sha256','source_sha256']:
    assert recomputed[key]==configuration[key],key
run('kia_validation.py','--phase','reserved','--development-measurements',measurements['development'],
    '--reserved-measurements',measurements['reserved'],'--output',work/'reserved_evaluation',
    '--configuration',root/'kia_calibration_configuration.json')
max_difference=0
for phase in ['development','reserved']:
    before=json.loads((cache/(phase+'_evaluation')/'predictions.json').read_text())
    after=json.loads((work/(phase+'_evaluation')/'predictions.json').read_text())
    for name in before:
        for method in ['frozen_mercedes_baseline',configuration['selected_method']]:
            a=before[name][method].get('estimated_kmh');b=after[name][method].get('estimated_kmh')
            if a is not None and b is not None:max_difference=max(max_difference,abs(a-b))
assert max_difference<.001,max_difference
out=Path('/kaggle/working/kia_validation')
video_dir=unique('KiaSportage_31.MP4').parent
for item in plan['input_videos']:
    with (video_dir/item['video']).open('rb') as stream:
        assert hashlib.file_digest(stream,'sha256').hexdigest()==item['sha256'],item['video']
run('publish_kia_report.py','--development-evaluation',work/'development_evaluation',
    '--reserved-evaluation',work/'reserved_evaluation','--development-measurements',measurements['development'],
    '--reserved-measurements',measurements['reserved'],'--configuration',root/'kia_calibration_configuration.json',
    '--output',out,'--video-dir',video_dir,'--mercedes-comparison',cache/'mercedes_regression')
shutil.copytree(cache/'development_initial',out/'development_initial')
write(out/'reproduction_verification.json',{'maximum_local_vs_kaggle_difference_kmh':max_difference,
      'fixed_registration_caches_used':True,'configuration_frozen_before_reserved_errors':True,
      'all_33_input_video_hashes_match_original_plan':True,
      '12_mathematical_and_exclusion_tests_passed':True})
write(out/'runtime.json',{'python':sys.version,'source_sha256':{
    name:hashlib.sha256((root/name).read_bytes()).hexdigest() for name in sources}})
assert len(list((out/'videos').glob('*/calibration.mp4')))==33
assert len(list((out/'videos').glob('*/speed.mp4')))==33
print('Private Kia reproduction and all 66 videos completed.')
